# Standing Dead Tree Segmentation in Forest Aerial Images

A comparative computer vision study on segmenting standing dead trees from aerial forest imagery, evaluating classical machine learning and modern deep learning approaches under real-world conditions.

## Project Overview
This project addresses the problem of **standing dead tree segmentation** in aerial forest images, a task with practical significance for forest management, ecological monitoring, and environmental assessment.  
The dataset consists of high-resolution aerial imagery with RGB and near-infrared (NIR) channels, paired with pixel-level ground-truth masks.

Segmenting dead trees in aerial scenes is challenging due to complex backgrounds, varying illumination, and subtle visual differences between living and dead vegetation.  
To systematically study these challenges, I designed and implemented a unified segmentation pipeline that compares multiple methods spanning traditional machine learning and state-of-the-art deep learning architectures.

## Project Goals
The project is driven by three core objectives:
1. Design a consistent data preprocessing and evaluation pipeline for aerial image segmentation.
2. Implement and benchmark a diverse set of segmentation models with different computational and representational characteristics.
3. Analyze performance trade-offs in terms of segmentation accuracy, robustness, and efficiency across methods.

## Methods Evaluated
The following segmentation approaches are implemented and compared:

- **Support Vector Machine (SVM)**  
  A traditional machine learning baseline using handcrafted features to establish a non-deep-learning reference.

- **UNet**  
  A convolutional encoder–decoder architecture widely used for fine-grained image segmentation.

- **DeepLabV3 (ResNet backbone)**  
  A state-of-the-art semantic segmentation model leveraging atrous convolution and spatial pyramid pooling for multi-scale context modeling.

- **Fast-SCNN**  
  A lightweight segmentation network designed for efficient inference in resource-constrained settings.

- **ADA-Net**  
  An attention-guided domain adaptation network tailored for aerial imagery segmentation.

All methods are evaluated under a unified experimental setup to ensure fair comparison.

The remainder of this project details dataset preparation, model implementation, quantitative evaluation, qualitative visualization, and comparative analysis of the above approaches.

## Environment Setup

The project is implemented using Python and PyTorch and is designed to run in a GPU-enabled environment.  
All experiments assume access to standard deep learning dependencies and sufficient memory for training and evaluation of segmentation models.

The execution environment supports loading datasets, saving trained model checkpoints, and storing evaluation outputs in a structured workspace, enabling reproducible experimentation across different systems.

In [ ]:
# Mount Google Drive to access dataset and output files
from google.colab import drive
drive.mount('/content/drive')

## Dataset Preparation

The dataset is extracted and organized into a structured workspace to ensure that all image data and corresponding segmentation masks are accessible for preprocessing, training, and evaluation.

In [ ]:
import zipfile
import os


# Define the path to the ZIP file in Google Drive
zip_path = "/content/drive/MyDrive/COMP9517/archive.zip"

# Define the destination path to extract the contents
extract_path = "/content/COMP9517"


# Extract the ZIP archive to the specified directory
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Archive extracted to:", extract_path)

### Dataset Verification

The dataset structure and contents are verified to ensure that all image and mask files are correctly organized and accessible before preprocessing and model training.

In [ ]:
# Walk through the extracted dataset directory and list its structure
for root, dirs, files in os.walk("/content/COMP9517"):
    print("Directory:", root)
    for d in dirs:
        print("  Subdirectory:", d)

# Print only the first 5 files in each directory to avoid long outputs
    for f in files[:5]:
        print("  File:", f)
    print("----------")

### Data Visualization and Sanity Check

Sample RGB images, near-infrared (NIR) channels, and corresponding ground-truth masks are visualized to verify correct data alignment and labeling.  
This qualitative inspection ensures that image channels and segmentation masks are consistent before model training.

In [ ]:
import cv2
import matplotlib.pyplot as plt

# Define paths to the input images
rgb_path = "/content/COMP9517/USA_segmentation/RGB_images/RGB_ar037_2019_n_06_04_0.png"
nrg_path = "/content/COMP9517/USA_segmentation/NRG_images/NRG_ar037_2019_n_06_04_0.png"
mask_path = "/content/COMP9517/USA_segmentation/masks/mask_ar037_2019_n_06_04_0.png"

# Load RGB and NRG images using OpenCV (in BGR format by default)
rgb = cv2.imread(rgb_path, cv2.IMREAD_COLOR)
nrg = cv2.imread(nrg_path, cv2.IMREAD_COLOR)

# Load ground truth mask in grayscale
mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

# Convert BGR to RGB for correct visualization in matplotlib
rgb = cv2.cvtColor(rgb, cv2.COLOR_BGR2RGB)
nrg = cv2.cvtColor(nrg, cv2.COLOR_BGR2RGB)

# Extract the NIR channel (assumed to be stored in the red channel of NRG image)
nir = nrg[:, :, 0]

# Visualize the RGB image, NIR channel, and segmentation mask
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.imshow(rgb)
plt.title("RGB Image")

plt.subplot(1, 3, 2)
plt.imshow(nir, cmap='gray')
plt.title("NIR Channel")

plt.subplot(1, 3, 3)
plt.imshow(mask, cmap='gray')
plt.title("Segmentation Mask")

plt.tight_layout()
plt.show()

### Training and Validation Split

The dataset is partitioned into training and validation subsets using a fixed split ratio to ensure reliable performance evaluation.  
Image–mask pairs are constructed based on filename alignment to prevent label mismatches, and the resulting splits are stored for reproducible model training.

In [ ]:
import os
import random
from glob import glob
import pandas as pd

# Set random seed for reproducibility
random.seed(42)

# Define dataset root and subdirectories
root_dir = "/content/COMP9517/USA_segmentation"
rgb_dir = os.path.join(root_dir, "RGB_images")
nrg_dir = os.path.join(root_dir, "NRG_images")
mask_dir = os.path.join(root_dir, "masks")

# Define directory to save output split files
save_dir = "/content/COMP9517"
os.makedirs(save_dir, exist_ok=True)

# Find all RGB image files
rgb_files = sorted(glob(os.path.join(rgb_dir, "*.png")))

# Helper function to extract a unique identifier from filename
# Example: 'RGB_ar037_2019_n_06_04_0.png' -> 'ar037_2019_n_06_04_0'
def extract_key(path):
    return os.path.basename(path).replace("RGB_", "").replace(".png", "")

# Match each RGB image with its corresponding NRG image and mask
data_pairs = []
for rgb_path in rgb_files:
    key = extract_key(rgb_path)
    mask_path = os.path.join(mask_dir, f"mask_{key}.png")
    nrg_path = os.path.join(nrg_dir, f"NRG_{key}.png")
    if os.path.exists(mask_path) and os.path.exists(nrg_path):
        data_pairs.append((rgb_path, nrg_path, mask_path))

# Output total number of matched triplets
print("Total image-mask pairs found:", len(data_pairs))

# Shuffle the data and split into 80% train, 20% validation
random.shuffle(data_pairs)
split_idx = int(0.8 * len(data_pairs))
train_pairs = data_pairs[:split_idx]
val_pairs = data_pairs[split_idx:]

print("Training set size:", len(train_pairs))
print("Validation set size:", len(val_pairs))

# === Save image-mask paths as .txt files ===
def save_txt(pairs, filepath):
    with open(filepath, 'w') as f:
        for rgb, nrg, mask in pairs:
            f.write(f"{rgb},{nrg},{mask}\n")

save_txt(train_pairs, os.path.join(save_dir, "train.txt"))
save_txt(val_pairs, os.path.join(save_dir, "val.txt"))
print("Saved train.txt and val.txt")

# === Save same pairs as .csv files (optional) for easier inspection ===
df_train = pd.DataFrame(train_pairs, columns=["RGB", "NRG", "Mask"])
df_val = pd.DataFrame(val_pairs, columns=["RGB", "NRG", "Mask"])
df_train.to_csv(os.path.join(save_dir, "train.csv"), index=False)
df_val.to_csv(os.path.join(save_dir, "val.csv"), index=False)
print("Saved train.csv and val.csv")

### Dataset and Data Loading

A custom PyTorch dataset is implemented to support flexible loading of aerial images and corresponding segmentation masks.  
The data pipeline accommodates both RGB and RGB+NIR inputs, applies consistent preprocessing to images and masks, and enables efficient batch loading for model training and validation.

In [ ]:
from torch.utils.data import Dataset, DataLoader
import torch
import numpy as np
import cv2
import os

# Custom dataset for loading RGB + optional NIR + mask triplets
class DeadTreeDataset(Dataset):
    def __init__(self, data_pairs, use_nir=False, resize=(384, 384)):
        self.data_pairs = data_pairs                  # List of (RGB, NRG, mask) paths
        self.use_nir = use_nir                        # Whether to include NIR as 4th channel
        self.resize = resize                          # Resize all images to this resolution (width, height)

    def __len__(self):
        return len(self.data_pairs)

    def __getitem__(self, idx):
        rgb_path, nrg_path, mask_path = self.data_pairs[idx]

        # Load RGB image, convert from BGR to RGB, and resize
        rgb = cv2.imread(rgb_path, cv2.IMREAD_COLOR)
        rgb = cv2.cvtColor(rgb, cv2.COLOR_BGR2RGB)
        rgb = cv2.resize(rgb, self.resize)

        if self.use_nir:
            # Load NRG image, convert to RGB, extract NIR channel (assumed in red)
            nrg = cv2.imread(nrg_path, cv2.IMREAD_COLOR)
            nrg = cv2.cvtColor(nrg, cv2.COLOR_BGR2RGB)
            nrg = cv2.resize(nrg, self.resize)
            nir = nrg[:, :, 0]  # Use red channel as NIR
            nir = np.expand_dims(nir, axis=2)
            image = np.concatenate([rgb, nir], axis=2)  # Result: H x W x 4
        else:
            image = rgb  # Use only RGB channels: H x W x 3

        # Normalize to [0, 1] and rearrange to C x H x W
        image = image.astype(np.float32) / 255.0
        image = np.transpose(image, (2, 0, 1))

        # Load segmentation mask, resize, and binarize
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        mask = cv2.resize(mask, self.resize)
        mask = (mask > 0).astype(np.float32)
        mask = np.expand_dims(mask, axis=0)  # Add channel dimension

        return torch.tensor(image, dtype=torch.float32), torch.tensor(mask, dtype=torch.float32)

# Set batch size and target input size for the model
batch_size = 8
input_size = (384, 384)  # Image size: (width, height)

# Create dataset instances for training and validation
train_dataset = DeadTreeDataset(train_pairs, use_nir=True, resize=input_size)
val_dataset = DeadTreeDataset(val_pairs, use_nir=True, resize=input_size)

# Create DataLoaders for batching and shuffling
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

# Check the output shape of one batch
for img, mask in train_loader:
    print("Image shape:", img.shape)  # Should be (B, 4, H, W) if use_nir=True
    print("Mask shape:", mask.shape)  # Should be (B, 1, H, W)
    break

## Dataset Preparation Summary

A complete data preparation pipeline is established to support reliable and reproducible segmentation experiments.  
The dataset is organized, verified, and partitioned into training and validation subsets, with careful alignment between aerial images and pixel-level segmentation masks.

A custom data loading abstraction is implemented to handle multi-channel inputs (RGB and RGB+NIR) and corresponding labels, enabling consistent preprocessing and efficient batch loading during model training.

With the data pipeline in place, the project proceeds to model implementation and training.

## Model Training and Evaluation

### SVM-Based Segmentation

As a classical baseline, a Support Vector Machine (SVM)–based segmentation approach is implemented to provide a non-deep-learning reference for comparison.

The method operates at the pixel level by constructing feature vectors from RGB and near-infrared (NIR) information and training a linear classifier to distinguish dead tree pixels from background.  
To handle class imbalance between foreground and background regions, balanced sampling is applied during training.

Each image is processed independently, producing full-resolution segmentation masks that are evaluated against ground-truth labels.  
Performance is assessed using multiple metrics, including Dice score, Intersection-over-Union (IoU), accuracy, precision, and recall.  
In addition to quantitative evaluation, qualitative visualizations such as overlay maps and worst-case predictions are generated to analyze model behavior and failure modes.

This SVM-based approach establishes a lightweight and interpretable baseline, against which more advanced deep learning segmentation models are compared.

In [ ]:
import os
import numpy as np
import cv2
import random
import matplotlib.pyplot as plt
from tqdm import tqdm
import pandas as pd
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    jaccard_score, f1_score, accuracy_score,
    confusion_matrix, ConfusionMatrixDisplay,
    precision_score, recall_score
)
from IPython.display import Image, display

# ========== CONFIG ==========
# Define input directories
BASE_DIR = "/content/COMP9517/USA_segmentation"
RGB_DIR = os.path.join(BASE_DIR, "RGB_images")
NRG_DIR = os.path.join(BASE_DIR, "NRG_images")
MASK_DIR = os.path.join(BASE_DIR, "masks")

# Define output directories
OUTPUT_DIR = "/content/COMP9517/outputs/svm_LinearSVC"
PRED_DIR = os.path.join(OUTPUT_DIR, "predictions")
OVERLAY_DIR = os.path.join(OUTPUT_DIR, "overlays")
VIZ_DIR = os.path.join(OUTPUT_DIR, "visualizations")
CM_DIR = os.path.join(OUTPUT_DIR, "confusion_matrices")
WORST_DIR = os.path.join(OUTPUT_DIR, "worst_predictions")

# Create output directories if they do not exist
for path in [PRED_DIR, OVERLAY_DIR, VIZ_DIR, CM_DIR, WORST_DIR]:
    os.makedirs(path, exist_ok=True)

# ========== Functions ==========

# Compute evaluation metrics between ground truth and predicted masks
def compute_metrics(gt, pred):
    gt_flat = gt.flatten()
    pred_flat = pred.flatten()
    return {
        "dice": f1_score(gt_flat, pred_flat, zero_division=1),
        "iou": jaccard_score(gt_flat, pred_flat, zero_division=1),
        "accuracy": accuracy_score(gt_flat, pred_flat),
        "precision": precision_score(gt_flat, pred_flat, zero_division=1),
        "recall": recall_score(gt_flat, pred_flat, zero_division=1),
    }

# Load and prepare RGB+NIR image and its binary segmentation mask
def get_feature_mask(rgb_path, nrg_path, mask_path, resize=(384, 384)):
    rgb = cv2.cvtColor(cv2.resize(cv2.imread(rgb_path), resize), cv2.COLOR_BGR2RGB)
    nrg = cv2.resize(cv2.imread(nrg_path), resize)
    nir = nrg[:, :, 0]  # Assume NIR is stored in red channel
    mask = cv2.resize(cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE), resize, interpolation=cv2.INTER_NEAREST)
    mask = (mask > 127).astype(np.uint8)
    return np.dstack((rgb, nir)), mask  # Return 4-channel image and binary mask

# Train a Linear SVM using a balanced subset of pixels from a single image
def train_linear_svm(image, mask, sample_size=1000):
    X = image.reshape(-1, 4)
    y = mask.flatten()
    pos_idx, neg_idx = np.where(y == 1)[0], np.where(y == 0)[0]
    if len(pos_idx) == 0 or len(neg_idx) == 0:
        return None  # Cannot train if only one class exists
    pos_sample = np.random.choice(pos_idx, min(sample_size // 2, len(pos_idx)), replace=False)
    neg_sample = np.random.choice(neg_idx, min(sample_size // 2, len(neg_idx)), replace=False)
    idx = np.concatenate([pos_sample, neg_sample])
    clf = LinearSVC(C=1.0, max_iter=1000)
    clf.fit(X[idx], y[idx])
    return clf

# Predict the full-size image using resized input for faster inference
def predict_full_image_fast(model, image, target_shape=(384, 384), pred_shape=(192, 192)):
    resized = cv2.resize(image, pred_shape, interpolation=cv2.INTER_LINEAR)
    X_pred = resized.reshape(-1, 4)
    pred_small = model.predict(X_pred).reshape(pred_shape)
    return cv2.resize(pred_small.astype(np.uint8), target_shape, interpolation=cv2.INTER_NEAREST)

# Save RGB image with predicted mask overlaid in red
def save_overlay(rgb, pred_mask, filename):
    overlay = rgb.copy()
    overlay[pred_mask == 1] = [255, 0, 0]  # Red for predicted foreground
    blended = cv2.addWeighted(rgb, 0.6, overlay, 0.4, 0)
    plt.imsave(os.path.join(OVERLAY_DIR, f"{filename}_overlay.png"), blended)

# Visualize RGB image, GT mask, and prediction side-by-side
def predict_and_visualize(rgb_img, gt_mask, pred_mask, filename, output_dir, display_inline=False):
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    titles = ["RGB Image", "Ground Truth Mask", "Predicted Mask"]
    for ax, img, title in zip(axes, [rgb_img, gt_mask, pred_mask], titles):
        ax.imshow(img if img.ndim == 3 else img, cmap='gray')
        ax.set_title(title)
    plt.tight_layout()
    out_path = os.path.join(output_dir, f"{filename}.png")
    plt.savefig(out_path, dpi=80)
    plt.close()
    if display_inline:
        display(Image(filename=out_path))

# ========== Main ==========

# Run SVM segmentation on all RGB-NRG-mask triplets in batch
def run_svm_batch_evaluation():
    results = []
    all_preds, all_targets = [], []
    all_files = sorted([f for f in os.listdir(RGB_DIR) if f.endswith(".png")])

    for fname in tqdm(all_files, desc="SVM Batch Prediction", leave=True):
        base = fname.replace("RGB_", "").replace(".png", "")
        paths = {
            'rgb': os.path.join(RGB_DIR, f"RGB_{base}.png"),
            'nrg': os.path.join(NRG_DIR, f"NRG_{base}.png"),
            'mask': os.path.join(MASK_DIR, f"mask_{base}.png")
        }

        # Load image and binary mask
        image, mask = get_feature_mask(paths['rgb'], paths['nrg'], paths['mask'])

        # Train Linear SVM on the current image
        model = train_linear_svm(image, mask)
        if model is None:
            print(f"[Warning] Skipping {fname} due to only one class in training sample.")
            continue

        # Predict and save outputs
        pred_mask = predict_full_image_fast(model, image)
        np.save(os.path.join(PRED_DIR, f"{base}_pred.npy"), pred_mask)
        save_overlay(image[:, :, :3], pred_mask, base)
        predict_and_visualize(image[:, :, :3], mask, pred_mask, base, VIZ_DIR, display_inline=False)

        # Compute evaluation metrics
        metrics = compute_metrics(mask, pred_mask)
        metrics.update({"filename": fname, "error": 1 - metrics["accuracy"]})
        results.append(metrics)
        all_preds.append(pred_mask.flatten())
        all_targets.append(mask.flatten())

    # Save metrics to CSV
    df = pd.DataFrame(results)
    df.to_csv("/content/COMP9517/training_log_svm.csv", index=False)

    # Generate confusion matrix
    cm = confusion_matrix(np.concatenate(all_targets), np.concatenate(all_preds))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[0, 1])
    disp.plot(cmap=plt.cm.Blues)
    plt.title("Confusion Matrix")
    plt.savefig(os.path.join(CM_DIR, "confusion_matrix.png"))
    plt.close()

    # Plot top-10 dice and IoU scores
    df_sorted = df.sort_values("dice", ascending=False).reset_index(drop=True)
    for metric, title in [("dice", "Top 10 Dice Scores"), ("iou", "Top 10 IoU Scores")]:
        plt.figure(figsize=(10, 5))
        plt.bar(df_sorted.index[:10], df_sorted[metric][:10])
        plt.xticks(df_sorted.index[:10], df_sorted["filename"][:10], rotation=45, ha="right")
        plt.ylabel(metric.upper())
        plt.title(title)
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, f"top10_{metric}_scores.png"))
        plt.close()

    # Show worst 5 predictions based on highest error
    for item in sorted(results, key=lambda x: x["error"], reverse=True)[:5]:
        base = item["filename"].replace("RGB_", "").replace(".png", "")
        rgb = cv2.cvtColor(cv2.imread(os.path.join(RGB_DIR, f"RGB_{base}.png")), cv2.COLOR_BGR2RGB)
        gt_mask = cv2.resize(cv2.imread(os.path.join(MASK_DIR, f"mask_{base}.png"), cv2.IMREAD_GRAYSCALE), (384, 384), interpolation=cv2.INTER_NEAREST)
        gt_mask = (gt_mask > 127).astype(np.uint8)
        pred_mask = np.load(os.path.join(PRED_DIR, f"{base}_pred.npy"))
        predict_and_visualize(rgb, gt_mask, pred_mask, f"worst_{base}", WORST_DIR, display_inline=True)

    # Preview summary plots
    print("\n=== Preview ===")
    for img in ["confusion_matrix.png", "top10_dice_scores.png", "top10_iou_scores.png"]:
        path = os.path.join(OUTPUT_DIR if 'top10' in img else CM_DIR, img)
        if os.path.exists(path):
            print(f"Showing {img}")
            display(Image(filename=path))

# Run the batch SVM segmentation
run_svm_batch_evaluation()

## Model 2: UNet Segmentation

UNet is implemented as a convolutional neural network baseline for pixel-level segmentation of standing dead trees in aerial imagery.  
Its encoder–decoder structure with skip connections enables effective extraction of high-level semantic context while preserving fine-grained spatial details, which is critical for accurate tree boundary delineation.

The model is configured to accept multi-channel inputs combining RGB and near-infrared (NIR) information, allowing it to leverage spectral cues beyond visible light.  
Feature extraction is performed through stacked convolutional blocks, while progressive downsampling and upsampling operations capture context at multiple spatial scales.  
A final pointwise convolution followed by a sigmoid activation produces pixel-wise segmentation probabilities.

This UNet-based model serves as a strong deep learning baseline and provides a reference point for comparison with more advanced segmentation architectures evaluated in this project.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Double convolution block: Conv -> BN -> ReLU -> Conv -> BN -> ReLU
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)

# U-Net architecture for image segmentation
class UNet(nn.Module):
    def __init__(self, in_channels=4, out_channels=1):
        super().__init__()

        # Encoder: Downsampling path
        self.down1 = DoubleConv(in_channels, 64)
        self.pool1 = nn.MaxPool2d(2)

        self.down2 = DoubleConv(64, 128)
        self.pool2 = nn.MaxPool2d(2)

        self.down3 = DoubleConv(128, 256)
        self.pool3 = nn.MaxPool2d(2)

        self.down4 = DoubleConv(256, 512)
        self.pool4 = nn.MaxPool2d(2)

        # Bottleneck
        self.bottleneck = DoubleConv(512, 1024)

        # Decoder: Upsampling path with skip connections
        self.up4 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.conv4 = DoubleConv(1024, 512)  # 512 upsampled + 512 skip

        self.up3 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.conv3 = DoubleConv(512, 256)  # 256 upsampled + 256 skip

        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.conv2 = DoubleConv(256, 128)  # 128 upsampled + 128 skip

        self.up1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.conv1 = DoubleConv(128, 64)   # 64 upsampled + 64 skip

        # Final segmentation output layer
        self.final_conv = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        # Encoder path
        d1 = self.down1(x)
        d2 = self.down2(self.pool1(d1))
        d3 = self.down3(self.pool2(d2))
        d4 = self.down4(self.pool3(d3))

        # Bottleneck
        bottleneck = self.bottleneck(self.pool4(d4))

        # Decoder path with skip connections
        up4 = self.up4(bottleneck)
        up4 = torch.cat([up4, d4], dim=1)
        up4 = self.conv4(up4)

        up3 = self.up3(up4)
        up3 = torch.cat([up3, d3], dim=1)
        up3 = self.conv3(up3)

        up2 = self.up2(up3)
        up2 = torch.cat([up2, d2], dim=1)
        up2 = self.conv2(up2)

        up1 = self.up1(up2)
        up1 = torch.cat([up1, d1], dim=1)
        up1 = self.conv1(up1)

        # Output: segmentation probability map (1 channel with sigmoid)
        return torch.sigmoid(self.final_conv(up1))

# Instantiate the U-Net model with 4 input channels and 1 output channel
model = UNet(in_channels=4, out_channels=1)

# Print model architecture
print(model)

# Test model with dummy input tensor
dummy_input = torch.randn(1, 4, 384, 384)  # (batch_size, channels, height, width)
output = model(dummy_input)

# Print output shape to verify dimension (expected: [1, 1, 384, 384])
print("Output shape:", output.shape)

# Count and print the total number of model parameters
total_params = sum(p.numel() for p in model.parameters())
print("Total parameters:", total_params)

### UNet Training and Evaluation

The UNet model is trained using a supervised segmentation pipeline tailored for multi-channel aerial imagery.  
Input images combine RGB and near-infrared (NIR) information, while pixel-level binary masks serve as training targets.

To optimize segmentation performance, a composite loss function is adopted that combines binary cross-entropy with Dice loss.  
This design balances pixel-wise classification accuracy with region-level overlap, which is particularly important for segmenting irregular and sparse tree structures.

Model performance is evaluated using multiple metrics, including Dice coefficient, Intersection-over-Union (IoU), and pixel accuracy.  
The best-performing model checkpoint is selected based on validation Dice score to ensure robust generalization.

This training configuration establishes UNet as a strong deep learning baseline, providing a reliable reference for comparison with more advanced segmentation architectures.

In [ ]:
import os
import time
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
from tqdm import tqdm
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

# ----------------------------
# Custom Dataset for segmentation (expects 4-channel .npy images)
# ----------------------------
class SegmentationDataset(Dataset):
    def __init__(self, image_paths, mask_paths, img_size=(384, 384)):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.img_size = img_size

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        mask_path = self.mask_paths[idx]

        image = cv2.imread(img_path, cv2.IMREAD_UNCHANGED)  # Expected: 4-channel image
        image = cv2.resize(image, self.img_size)
        image = image.astype(np.float32) / 255.0
        image = torch.from_numpy(image).permute(2, 0, 1)  # Convert to (C, H, W)

        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        mask = cv2.resize(mask, self.img_size, interpolation=cv2.INTER_NEAREST)
        mask = (mask > 127).astype(np.float32)
        mask = torch.from_numpy(mask).unsqueeze(0)

        return image, mask

# ----------------------------
# U-Net Model Definition
# ----------------------------
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.double_conv(x)

class UNet(nn.Module):
    def __init__(self, in_channels=4, out_channels=1):
        super().__init__()
        self.down1 = DoubleConv(in_channels, 64)
        self.pool1 = nn.MaxPool2d(2)
        self.down2 = DoubleConv(64, 128)
        self.pool2 = nn.MaxPool2d(2)
        self.down3 = DoubleConv(128, 256)
        self.pool3 = nn.MaxPool2d(2)
        self.down4 = DoubleConv(256, 512)
        self.pool4 = nn.MaxPool2d(2)

        self.bottleneck = DoubleConv(512, 1024)

        self.up4 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.conv4 = DoubleConv(1024, 512)
        self.up3 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.conv3 = DoubleConv(512, 256)
        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.conv2 = DoubleConv(256, 128)
        self.up1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.conv1 = DoubleConv(128, 64)

        self.final_conv = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        d1 = self.down1(x)
        d2 = self.down2(self.pool1(d1))
        d3 = self.down3(self.pool2(d2))
        d4 = self.down4(self.pool3(d3))

        bn = self.bottleneck(self.pool4(d4))

        up4 = self.up4(bn)
        up4 = self.conv4(torch.cat([up4, d4], dim=1))
        up3 = self.up3(up4)
        up3 = self.conv3(torch.cat([up3, d3], dim=1))
        up2 = self.up2(up3)
        up2 = self.conv2(torch.cat([up2, d2], dim=1))
        up1 = self.up1(up2)
        up1 = self.conv1(torch.cat([up1, d1], dim=1))

        return torch.sigmoid(self.final_conv(up1))

# ----------------------------
# Evaluation Metrics
# ----------------------------
def dice_coefficient(pred, target, smooth=1e-6):
    pred_flat = pred.view(-1)
    target_flat = target.view(-1)
    intersection = torch.sum(pred_flat * target_flat)
    return (2.0 * intersection + smooth) / (torch.sum(pred_flat) + torch.sum(target_flat) + smooth)

def calculate_iou(pred_mask, true_mask, threshold=0.5):
    pred_binary = (pred_mask > threshold).float()
    true_binary = true_mask.float()
    intersection = torch.sum(pred_binary * true_binary)
    union = torch.sum(pred_binary) + torch.sum(true_binary) - intersection
    return (intersection / union).item() if union > 0 else 1.0

def calculate_accuracy(pred_mask, true_mask, threshold=0.5):
    pred_binary = (pred_mask > threshold).float()
    true_binary = true_mask.float()
    correct = torch.sum(pred_binary == true_binary)
    total = torch.numel(pred_binary)
    return (correct / total).item()

# ----------------------------
# Loss Function: BCE + Dice
# ----------------------------
def combined_loss(pred, target, alpha=0.5):
    bce = F.binary_cross_entropy(pred, target)
    dice = 1 - dice_coefficient(pred, target)
    return alpha * bce + (1 - alpha) * dice

# ----------------------------
# Model Training Function
# ----------------------------
def train_segmentation_model(model, train_loader, val_loader, device, model_name="unet", epochs=20, lr=1e-4):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    # Initialize tracking
    best_dice = 0.0
    best_acc = 0.0
    best_iou = 0.0

    best_dice_epoch = 0
    best_acc_epoch = 0
    best_iou_epoch = 0

    start_time = time.time()
    history = []

    # Create model save directory
    save_path = f"/content/COMP9517/checkpoints/{model_name}_best.pth"
    os.makedirs(os.path.dirname(save_path), exist_ok=True)

    for epoch in range(1, epochs + 1):
        epoch_start = time.time()
        model.train()
        train_loss = 0.0
        print(f"\nEpoch {epoch}/{epochs}")

        # Training loop
        with tqdm(train_loader, desc="Training") as pbar:
            for images, masks in pbar:
                images = images.to(device)
                masks = masks.to(device)
                optimizer.zero_grad()
                outputs = model(images)
                loss = combined_loss(outputs, masks)
                loss.backward()
                optimizer.step()
                train_loss += loss.item()
                pbar.set_postfix({"loss": f"{loss.item():.2f}"})

        # Validation loop
        model.eval()
        val_loss = 0.0
        total_iou = 0.0
        total_dice = 0.0
        total_acc = 0.0

        with torch.no_grad():
            for images, masks in tqdm(val_loader, desc="Validation"):
                images = images.to(device)
                masks = masks.to(device)
                outputs = model(images)
                loss = combined_loss(outputs, masks)
                val_loss += loss.item()
                total_iou += calculate_iou(outputs, masks)
                total_dice += dice_coefficient(outputs, masks)
                total_acc += calculate_accuracy(outputs, masks)

        # Epoch summary
        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)
        avg_iou = total_iou / len(val_loader)
        avg_dice = total_dice / len(val_loader)
        avg_acc = total_acc / len(val_loader)
        epoch_time = time.time() - epoch_start

        print(f"\nTrain Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | IoU: {avg_iou:.4f} | Dice: {avg_dice:.4f} | Accuracy: {avg_acc:.4f} | Time: {epoch_time:.1f}s")

        # Save best model (based on Dice)
        if avg_dice > best_dice:
            best_dice = avg_dice
            best_dice_epoch = epoch
            torch.save(model.state_dict(), save_path)
            print("Saved best model (based on Dice).")
        else:
            print("No improvement in Dice.")

        # Track best accuracy and IoU separately
        if avg_acc > best_acc:
            best_acc = avg_acc
            best_acc_epoch = epoch

        if avg_iou > best_iou:
            best_iou = avg_iou
            best_iou_epoch = epoch

        # Save history
        history.append({
            "epoch": epoch,
            "train_loss": avg_train_loss,
            "val_loss": avg_val_loss,
            "iou": avg_iou,
            "dice": avg_dice,
            "accuracy": avg_acc,
            "time_sec": round(epoch_time, 2)
        })

    # Save training log
    total_time = time.time() - start_time
    df = pd.DataFrame(history)
    os.makedirs("/content/COMP9517", exist_ok=True)
    log_path = f"/content/COMP9517/training_log_{model_name}.csv"
    df.to_csv(log_path, index=False)

    # Final summary
    print(f"\nTraining finished. Log saved to {log_path}")
    print(f"Best validation Dice: {best_dice:.4f} (Epoch {best_dice_epoch})")
    print(f"Best validation Accuracy: {best_acc:.4f} (Epoch {best_acc_epoch})")
    print(f"Best validation IoU: {best_iou:.4f} (Epoch {best_iou_epoch})")
    print(f"Total training time: {total_time:.1f} seconds")

    return df

# ----------------------------
# Main execution
# ----------------------------
if __name__ == "__main__":
    # Set random seed for reproducibility
    torch.manual_seed(42)
    np.random.seed(42)

    # Define dataset paths
    RGB_DIR = "/content/COMP9517/USA_segmentation/RGB_images"
    NRG_DIR = "/content/COMP9517/USA_segmentation/NRG_images"
    MASK_DIR = "/content/COMP9517/USA_segmentation/masks"

    # Load and save combined 4-channel (RGB + NIR) as .npy
    image_paths = []
    mask_paths = []
    for fname in sorted(os.listdir(MASK_DIR)):
        mask_path = os.path.join(MASK_DIR, fname)
        base = fname.replace("mask_", "").replace(".png", "")
        rgb_path = os.path.join(RGB_DIR, f"RGB_{base}.png")
        nrg_path = os.path.join(NRG_DIR, f"NRG_{base}.png")
        if os.path.exists(rgb_path) and os.path.exists(nrg_path):
            rgb = cv2.imread(rgb_path)
            nrg = cv2.imread(nrg_path)
            nir = nrg[:, :, 0:1]  # NIR as single channel
            combined = np.concatenate((rgb, nir), axis=2)  # Shape (H, W, 4)
            save_path = f"/content/COMP9517/combined/{base}.npy"
            os.makedirs(os.path.dirname(save_path), exist_ok=True)
            np.save(save_path, combined)
            image_paths.append(save_path)
            mask_paths.append(mask_path)

    # Split into training and validation sets (80/20)
    total = len(image_paths)
    indices = np.random.permutation(total)
    split = int(0.8 * total)
    train_idx, val_idx = indices[:split], indices[split:]
    train_imgs = [image_paths[i] for i in train_idx]
    train_masks = [mask_paths[i] for i in train_idx]
    val_imgs = [image_paths[i] for i in val_idx]
    val_masks = [mask_paths[i] for i in val_idx]

    # Define dataset class for .npy loading
    class NPZDataset(SegmentationDataset):
        def __getitem__(self, idx):
            image = np.load(self.image_paths[idx])
            image = cv2.resize(image, self.img_size)
            image = image.astype(np.float32) / 255.0
            image = torch.from_numpy(image).permute(2, 0, 1)
            mask = cv2.imread(self.mask_paths[idx], cv2.IMREAD_GRAYSCALE)
            mask = cv2.resize(mask, self.img_size, interpolation=cv2.INTER_NEAREST)
            mask = (mask > 127).astype(np.float32)
            mask = torch.from_numpy(mask).unsqueeze(0)
            return image, mask

    # Create dataloaders
    train_ds = NPZDataset(train_imgs, train_masks, img_size=(384, 384))
    val_ds = NPZDataset(val_imgs, val_masks, img_size=(384, 384))
    train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=2)

    # Select device (GPU if available)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # Initialize model and begin training
    model = UNet(in_channels=4, out_channels=1).to(device)
    train_segmentation_model(model, train_loader, val_loader, device, model_name="unet", epochs=20, lr=1e-4)

### UNet Training Dynamics and Convergence Analysis

Training dynamics are monitored through visualization of loss and performance metrics across epochs.  
Key indicators such as training and validation loss, Dice score, Intersection-over-Union (IoU), and pixel accuracy are tracked to assess model convergence and generalization behavior.

These visualizations provide insight into the stability of the optimization process and help identify potential issues such as underfitting or overfitting.  
The resulting training curves also facilitate fair qualitative comparison with other segmentation models evaluated in this project.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os
from IPython.display import display, Image

# Visualize training metrics from a CSV log file and save plots
def visualize_training_log(log_path="/content/COMP9517/training_log_unet.csv"):
    # Load training log CSV
    log = pd.read_csv(log_path)

    # Define and create output directory for plots
    output_dir = "/content/COMP9517/outputs/unet/plots"
    os.makedirs(output_dir, exist_ok=True)

    # Handle possible 'tensor(...)' formatting in 'dice' column (convert to float)
    if log["dice"].dtype == object and log["dice"].astype(str).str.contains("tensor").any():
        log["dice"] = log["dice"].apply(lambda x: float(str(x).split("(")[-1].split(",")[0]))

    # Helper function to create and save a plot
    def save_plot(x, y, ylabel, title, filename, color=None):
        plt.figure()
        plt.plot(x, y, label=ylabel, color=color)
        plt.xlabel("Epoch")
        plt.ylabel(ylabel)
        plt.title(title)
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, filename), bbox_inches='tight')
        plt.close()

    # Generate and save plots for each metric
    save_plot(log["epoch"], log["train_loss"],
              "Train Loss", "Train Loss over Epochs", "train_loss_curve.png")
    save_plot(log["epoch"], log["val_loss"],
              "Validation Loss", "Validation Loss over Epochs", "val_loss_curve.png")
    save_plot(log["epoch"], log["accuracy"],
              "Accuracy", "Validation Accuracy over Epochs", "accuracy_curve.png", "green")
    save_plot(log["epoch"], log["dice"],
              "Dice", "Validation Dice over Epochs", "dice_curve.png", "purple")
    save_plot(log["epoch"], log["iou"],
              "IoU", "Validation IoU over Epochs", "iou_curve.png", "orange")

    print("UNet training plots saved to:", output_dir)

    # Display all plots inline in notebook (if running interactively)
    print("\n[Preview of Training Plots]")
    for name in ["train_loss_curve.png", "val_loss_curve.png", "accuracy_curve.png", "dice_curve.png", "iou_curve.png"]:
        path = os.path.join(output_dir, name)
        if os.path.exists(path):
            print(f"Showing: {name}")
            display(Image(filename=path))

# Run visualization when the script is executed directly
if __name__ == "__main__":
    visualize_training_log("/content/COMP9517/training_log_unet.csv")

### UNet Validation Performance Analysis

The trained UNet model is evaluated on a held-out validation set using both quantitative metrics and qualitative inspection.  
Segmentation performance is assessed with multiple complementary metrics, including pixel accuracy, Dice score, Intersection-over-Union (IoU), precision, and recall, providing a comprehensive view of model behavior.

Beyond aggregate metrics, qualitative analysis is conducted by inspecting representative predictions and failure cases.  
Visualizations of correctly and incorrectly segmented samples help reveal common error patterns and highlight the strengths and limitations of the model in complex aerial scenes.

This evaluation establishes a reliable performance baseline for UNet and enables direct comparison with other segmentation approaches explored in this project.

In [ ]:
import os
import time
import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, precision_score, recall_score, f1_score, jaccard_score, accuracy_score
from IPython.display import display, Image
import random

# -------------------------------
# Load trained UNet model from checkpoint
# -------------------------------
model_name = "unet"
checkpoint_path = f"/content/COMP9517/checkpoints/{model_name}_best.pth"

# Define output directories for predictions and evaluation results
base_dir = f"/content/COMP9517/outputs/{model_name}"
pred_dir = os.path.join(base_dir, "predictions")
npy_dir = os.path.join(pred_dir, "npy")
worst_dir = os.path.join(pred_dir, "worst")
conf_matrix_dir = os.path.join(base_dir, "confusion_matrix")

# Ensure output folders exist
os.makedirs(npy_dir, exist_ok=True)
os.makedirs(worst_dir, exist_ok=True)
os.makedirs(conf_matrix_dir, exist_ok=True)

# Select device (GPU if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Initialize model and load weights
model = UNet(in_channels=4, out_channels=1)
model.load_state_dict(torch.load(checkpoint_path, map_location=device))
model.to(device)
model.eval()

# -------------------------------
# Pixel-wise accuracy calculation helper
# -------------------------------
def compute_accuracy(pred, target, threshold=0.5):
    pred = (pred > threshold).float()
    target = (target > 0.5).float()
    correct = (pred == target).float().sum()
    total = torch.numel(pred)
    return (correct / total).item()

# -------------------------------
# Evaluation loop over validation loader
# -------------------------------
start_time = time.time()
total_acc = total_dice = total_iou = total_precision = total_recall = 0
num_samples = 0
errors = []           # Store per-sample errors for worst-case selection
all_preds = []        # Flattened predictions for confusion matrix
all_targets = []      # Flattened ground truth for confusion matrix
sample_paths = []     # Store tensors for visualization

for idx, (image, mask) in enumerate(tqdm(val_loader, desc="UNet Evaluation")):
    image = image.to(device)
    mask = mask.to(device)

    with torch.no_grad():
        pred = model(image)
        pred_bin = (pred > 0.5).float()

    # Compute metrics for this sample
    acc = compute_accuracy(pred, mask)
    dice = f1_score(mask.view(-1).cpu().numpy(), pred_bin.view(-1).cpu().numpy(), zero_division=1)
    iou = jaccard_score(mask.view(-1).cpu().numpy(), pred_bin.view(-1).cpu().numpy(), zero_division=1)
    prec = precision_score(mask.view(-1).cpu().numpy(), pred_bin.view(-1).cpu().numpy(), zero_division=1)
    rec = recall_score(mask.view(-1).cpu().numpy(), pred_bin.view(-1).cpu().numpy(), zero_division=1)

    # Accumulate totals for averaging
    total_acc += acc
    total_dice += dice
    total_iou += iou
    total_precision += prec
    total_recall += rec
    num_samples += 1
    errors.append((idx, 1 - acc))  # Store sample index and error rate

    # Save predictions and masks as .npy for further analysis
    np.save(os.path.join(npy_dir, f"pred_{idx}.npy"), pred[0].squeeze().cpu().numpy())
    np.save(os.path.join(npy_dir, f"mask_{idx}.npy"), mask[0].squeeze().cpu().numpy())

    # Store flattened predictions/targets for confusion matrix
    all_preds.append(pred_bin.view(-1).cpu().numpy())
    all_targets.append(mask.view(-1).cpu().numpy())

    # Keep image, GT mask, and prediction for visualization
    sample_paths.append((image[0, :3].cpu(), mask[0].cpu(), pred[0].cpu()))

# -------------------------------
# Compute average metrics over all samples
# -------------------------------
avg_acc = total_acc / num_samples
avg_dice = total_dice / num_samples
avg_iou = total_iou / num_samples
avg_prec = total_precision / num_samples
avg_recall = total_recall / num_samples
total_time = time.time() - start_time

# Print evaluation summary
print(f"Total samples predicted: {num_samples}")
print(f"Average pixel-wise accuracy on validation set: {avg_acc:.4f}")
print(f"Average Dice: {avg_dice:.4f}")
print(f"Average IoU: {avg_iou:.4f}")
print(f"Average Precision: {avg_prec:.4f}")
print(f"Average Recall: {avg_recall:.4f}")
print(f"Total inference time: {total_time:.2f} seconds")
print(f"Prediction images saved in: {pred_dir}")

# -------------------------------
# Save evaluation metrics to CSV
# -------------------------------
eval_df = pd.DataFrame([{
    "Model": model_name.upper(),
    "Accuracy": avg_acc,
    "Dice": avg_dice,
    "IoU": avg_iou,
    "Precision": avg_prec,
    "Recall": avg_recall,
    "InferenceTimeSec": total_time
}])
eval_df.to_csv(os.path.join(base_dir, f"eval_results_{model_name}.csv"), index=False)

# -------------------------------
# Generate and save confusion matrix plot
# -------------------------------
all_preds_np = np.concatenate(all_preds)
all_targets_np = np.concatenate(all_targets)
cm = confusion_matrix(all_targets_np, all_preds_np)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Background", "Tree"])
disp.plot(cmap=plt.cm.Blues, values_format="d")
plt.title(f"Confusion Matrix - {model_name.upper()}", fontsize=14, pad=10)
plt.grid(False)
plt.subplots_adjust(left=0.25)
confusion_image_path = os.path.join(conf_matrix_dir, f"{model_name}_confusion_matrix.png")
plt.savefig(confusion_image_path, bbox_inches='tight')
plt.close()
print(f"Confusion matrix saved to {confusion_image_path}")
display(Image(filename=confusion_image_path))

# -------------------------------
# Visualize worst-case predictions (highest error rates)
# -------------------------------
worst_k = 5
print("\nWorst-case prediction samples (highest prediction error):")
worst_samples = sorted(errors, key=lambda x: x[1], reverse=True)[:worst_k]
for idx, err in worst_samples:
    img_tensor, gt_mask_tensor, pred_tensor = sample_paths[idx]
    rgb = img_tensor[:3].permute(1, 2, 0).numpy()
    rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min())
    gt_mask = gt_mask_tensor.squeeze().numpy()
    pred_mask = pred_tensor.squeeze().numpy()

    fig, axs = plt.subplots(1, 3, figsize=(15, 5))
    axs[0].imshow(rgb)
    axs[0].set_title("RGB Image", pad=10)
    axs[1].imshow(gt_mask, cmap="gray")
    axs[1].set_title("Ground Truth Mask", pad=10)
    axs[2].imshow(pred_mask > 0.5, cmap="gray")
    axs[2].set_title("Predicted Mask", pad=10)
    plt.tight_layout()
    path = os.path.join(worst_dir, f"worst_{idx}.png")
    plt.savefig(path)
    plt.close()
    display(Image(filename=path))

# -------------------------------
# Visualize random prediction samples
# -------------------------------
print("\nRandom prediction samples:")
random_indices = random.sample(range(len(sample_paths)), min(5, len(sample_paths)))
for idx in random_indices:
    img_tensor, gt_mask_tensor, pred_tensor = sample_paths[idx]
    rgb = img_tensor[:3].permute(1, 2, 0).numpy()
    rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min())
    gt_mask = gt_mask_tensor.squeeze().numpy()
    pred_mask = pred_tensor.squeeze().numpy()

    fig, axs = plt.subplots(1, 3, figsize=(15, 5))
    axs[0].imshow(rgb)
    axs[0].set_title("RGB Image", pad=10)
    axs[1].imshow(gt_mask, cmap="gray")
    axs[1].set_title("Ground Truth Mask", pad=10)
    axs[2].imshow(pred_mask > 0.5, cmap="gray")
    axs[2].set_title("Predicted Mask", pad=10)
    plt.tight_layout()
    plt.show()

## Model 3: DeepLabV3 Segmentation

DeepLabV3 is evaluated as a strong semantic segmentation baseline designed to capture multi-scale contextual information in complex scenes.  
By leveraging atrous convolution and spatial pyramid pooling, the model is well-suited for segmenting objects with varying sizes and irregular boundaries in aerial imagery.

The model employs a ResNet-based backbone pretrained on large-scale image data and is adapted for binary segmentation of standing dead trees.  
Training is conducted using a supervised segmentation pipeline, and performance is assessed using pixel accuracy, Dice score, and Intersection-over-Union (IoU) to enable direct comparison with previously evaluated models.

This DeepLabV3-based approach provides a high-capacity reference point within the model comparison, complementing both classical baselines and encoder–decoder architectures such as UNet.

In [ ]:
import os
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms
from torchvision.models.segmentation import DeepLabV3_ResNet50_Weights
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import pandas as pd
from torchvision.transforms import functional as TF

# ----------------------------
# Path configuration
# ----------------------------
DATA_DIR = "/content/COMP9517/USA_segmentation"
RGB_DIR = os.path.join(DATA_DIR, "RGB_images")
MASK_DIR = os.path.join(DATA_DIR, "masks")

# ----------------------------
# Custom Dataset: loads and resizes RGB images and masks
# ----------------------------
class ResizedTreeSegmentationDataset(Dataset):
    def __init__(self, rgb_dir, mask_dir, size=(360, 360)):
        self.rgb_dir = rgb_dir
        self.mask_dir = mask_dir
        self.filenames = sorted([f for f in os.listdir(rgb_dir) if f.endswith('.png')])
        self.size = size

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        fname = self.filenames[idx]
        rgb_path = os.path.join(self.rgb_dir, fname)
        mask_path = os.path.join(self.mask_dir, "mask_" + fname.replace("RGB_", ""))

        # Load RGB image and segmentation mask
        rgb = Image.open(rgb_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")

        # Resize both image and mask
        rgb = TF.resize(rgb, self.size)
        mask = TF.resize(mask, self.size, interpolation=Image.NEAREST)

        # Convert RGB to tensor [0,1]
        rgb = TF.to_tensor(rgb)

        # Convert mask to binary tensor
        mask = np.array(mask)
        mask = (mask > 127).astype(np.uint8)
        mask = torch.tensor(mask, dtype=torch.long)

        return rgb, mask

# ----------------------------
# Prepare dataset and DataLoaders
# ----------------------------
full_dataset = ResizedTreeSegmentationDataset(RGB_DIR, MASK_DIR, size=(360, 360))
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(full_dataset, [train_size, val_size])
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=2, shuffle=False)

# ----------------------------
# Model setup (DeepLabV3 with ResNet-50 backbone)
# ----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

weights = DeepLabV3_ResNet50_Weights.DEFAULT
model = models.segmentation.deeplabv3_resnet50(weights=weights)
model.classifier[4] = nn.Conv2d(256, 2, kernel_size=1)  # Adjust for 2 classes
model = model.to(device)

# Loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# ----------------------------
# Paths for saving model and training log
# ----------------------------
model_name = "deeplabv3"
save_path = f"/content/COMP9517/checkpoints/{model_name}_best.pth"
log_path = f"/content/COMP9517/training_log_{model_name}.csv"
os.makedirs(os.path.dirname(save_path), exist_ok=True)

# ----------------------------
# Training loop
# ----------------------------
num_epochs = 20
best_dice = 0.0
best_iou = 0.0
best_acc = 0.0
best_dice_epoch = 0
best_iou_epoch = 0
best_acc_epoch = 0
log = []

for epoch in range(1, num_epochs + 1):
    start_time = time.time()
    model.train()
    train_loss = 0

    # Training phase
    train_bar = tqdm(train_loader, desc=f"\nEpoch {epoch}/{num_epochs}")
    for images, masks in train_bar:
        images, masks = images.to(device), masks.to(device)
        outputs = model(images)["out"]
        loss = criterion(outputs, masks)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        train_bar.set_description(f"Epoch {epoch}/{num_epochs}")
        train_bar.set_postfix(loss=loss.item())

    avg_train_loss = train_loss / len(train_loader)

    # Validation phase
    model.eval()
    val_loss = 0
    total_correct = 0
    total_pixels = 0
    total_iou = 0
    total_dice = 0
    val_bar = tqdm(val_loader, desc="Validation")
    with torch.no_grad():
        for images, masks in val_bar:
            images, masks = images.to(device), masks.to(device)
            outputs = model(images)["out"]
            loss = criterion(outputs, masks)
            val_loss += loss.item()

            preds = torch.argmax(outputs, dim=1)

            # Accuracy
            correct = (preds == masks).sum().item()
            total_correct += correct
            total_pixels += torch.numel(preds)

            # IoU and Dice
            intersection = ((preds == 1) & (masks == 1)).sum().item()
            union = ((preds == 1) | (masks == 1)).sum().item()
            iou = intersection / union if union > 0 else 0
            dice = (2 * intersection) / (preds.eq(1).sum().item() + masks.eq(1).sum().item() + 1e-8)
            total_iou += iou
            total_dice += dice

    avg_val_loss = val_loss / len(val_loader)
    avg_acc = total_correct / total_pixels
    avg_iou = total_iou / len(val_loader)
    avg_dice = total_dice / len(val_loader)
    elapsed = time.time() - start_time

    print()
    print(f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | IoU: {avg_iou:.4f} | Dice: {avg_dice:.4f} | Accuracy: {avg_acc:.4f} | Time: {elapsed:.1f}s")

    # Save best model based on Dice score
    if avg_dice > best_dice:
        best_dice = avg_dice
        best_dice_epoch = epoch
        torch.save(model.state_dict(), save_path)
        print("Saved best model (based on Dice).\n")
    else:
        print("No improvement in Dice.\n")

    # Track best IoU and Accuracy separately
    if avg_iou > best_iou:
        best_iou = avg_iou
        best_iou_epoch = epoch
    if avg_acc > best_acc:
        best_acc = avg_acc
        best_acc_epoch = epoch

    # Log metrics
    log.append({
        "Epoch": epoch,
        "Train Loss": avg_train_loss,
        "Val Loss": avg_val_loss,
        "IoU": avg_iou,
        "Dice": avg_dice,
        "Accuracy": avg_acc,
        "Time": elapsed
    })

# ----------------------------
# Save training log and print summary
# ----------------------------
total_time = sum(entry["Time"] for entry in log)
log_df = pd.DataFrame(log)
log_df.to_csv(log_path, index=False)

print()
print(f"Training finished. Log saved to {log_path}")
print(f"Best validation Dice: {best_dice:.4f} (Epoch {best_dice_epoch})")
print(f"Best validation Accuracy: {best_acc:.4f} (Epoch {best_acc_epoch})")
print(f"Best validation IoU: {best_iou:.4f} (Epoch {best_iou_epoch})")
print(f"Total training time: {total_time:.1f} seconds")

### DeepLabV3 Training Dynamics and Convergence Analysis

Training behavior of the DeepLabV3 model is analyzed through visualization of loss and segmentation performance metrics across epochs.  
Key indicators such as training and validation loss, Dice score, Intersection-over-Union (IoU), and pixel accuracy are tracked to evaluate optimization stability and convergence.

These training curves provide insight into how effectively the model learns multi-scale contextual features and facilitate qualitative comparison with other segmentation approaches, including UNet and the classical SVM baseline.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os
from IPython.display import display, Image

def visualize_training_log(log_path="/content/COMP9517/training_log_deeplabv3.csv"):

    # Load training metrics from CSV
    log = pd.read_csv(log_path)

    # Ensure output directory exists for saving plots
    output_dir = "/content/COMP9517/outputs/deeplabv3/plots"
    os.makedirs(output_dir, exist_ok=True)

    # Convert 'Dice' column to float if stored as stringified tensor (fallback case)
    if log["Dice"].dtype == object and log["Dice"].astype(str).str.contains("tensor").any():
        log["Dice"] = log["Dice"].apply(lambda x: float(str(x).split("(")[-1].split(",")[0]))

    # Helper function to create and save a single metric plot
    def save_plot(x, y, ylabel, title, filename, color=None):
        plt.figure()
        plt.plot(x, y, label=ylabel, color=color)
        plt.xlabel("Epoch")
        plt.ylabel(ylabel)
        plt.title(title)
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, filename), bbox_inches='tight')
        plt.close()

    # Generate and save plots for each metric
    save_plot(log["Epoch"], log["Train Loss"],
              "Train Loss", "Train Loss over Epochs", "train_loss_curve.png")
    save_plot(log["Epoch"], log["Val Loss"],
              "Validation Loss", "Validation Loss over Epochs", "val_loss_curve.png")
    save_plot(log["Epoch"], log["Accuracy"],
              "Accuracy", "Validation Accuracy over Epochs", "accuracy_curve.png", "green")
    save_plot(log["Epoch"], log["Dice"],
              "Dice", "Validation Dice over Epochs", "dice_curve.png", "purple")
    save_plot(log["Epoch"], log["IoU"],
              "IoU", "Validation IoU over Epochs", "iou_curve.png", "orange")

    print("DeepLabV3 training plots saved to:", output_dir)

    # Optionally display plots inline (useful in Jupyter/Colab)
    print("\n[Preview of Training Plots]")
    for name in ["train_loss_curve.png", "val_loss_curve.png", "accuracy_curve.png", "dice_curve.png", "iou_curve.png"]:
        path = os.path.join(output_dir, name)
        if os.path.exists(path):
            print(f"Showing: {name}")
            display(Image(filename=path))

# Run only when executed directly (prevents auto-run on import)
if __name__ == "__main__":
    visualize_training_log("/content/COMP9517/training_log_deeplabv3.csv")

### DeepLabV3 Validation Performance Analysis

The trained DeepLabV3 model is evaluated on a held-out validation set to assess its segmentation performance on aerial forest imagery.  
Evaluation is conducted using a comprehensive set of metrics, including pixel accuracy, Dice score, Intersection-over-Union (IoU), precision, and recall, providing both region-level and pixel-level performance insights.

In addition to aggregated metrics, qualitative analysis is performed by inspecting representative predictions and failure cases.  
Visualization of correct and incorrect segmentations helps reveal how effectively the model captures tree structures and where it struggles in complex background regions.

This validation analysis establishes DeepLabV3 as a high-capacity semantic segmentation baseline and enables direct comparison with UNet and classical SVM-based approaches.

In [ ]:
import os
import time
import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    precision_score, recall_score, f1_score, jaccard_score, accuracy_score
)
from IPython.display import display, Image
from torchvision import models
import torch.nn as nn
import random

from torch.utils.data import Dataset, DataLoader
from PIL import Image as PILImage
from torchvision.transforms import functional as TF

# ===============================
# Dataset: loads and resizes RGBA images and binary masks
# ===============================
class ResizedTreeSegmentationDataset(Dataset):
    def __init__(self, rgb_dir, mask_dir, size=(360, 360)):
        self.rgb_dir = rgb_dir
        self.mask_dir = mask_dir
        self.filenames = sorted([f for f in os.listdir(rgb_dir) if f.endswith('.png')])
        self.size = size

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        fname = self.filenames[idx]
        rgb_path = os.path.join(self.rgb_dir, fname)
        mask_path = os.path.join(self.mask_dir, "mask_" + fname.replace("RGB_", ""))

        # Load RGBA image (4 channels) and mask (grayscale)
        rgb = PILImage.open(rgb_path).convert("RGBA")  # 4-channel input
        mask = PILImage.open(mask_path).convert("L")   # binary mask

        # Resize both image and mask
        rgb = TF.resize(rgb, self.size)
        mask = TF.resize(mask, self.size, interpolation=PILImage.NEAREST)

        # Convert image to tensor and mask to binary tensor
        rgb = TF.to_tensor(rgb)
        mask = np.array(mask)
        mask = (mask > 127).astype(np.uint8)
        mask = torch.tensor(mask, dtype=torch.long)

        return rgb, mask

# ===============================
# Paths and validation loader setup
# ===============================
DATA_DIR = "/content/COMP9517/USA_segmentation"
RGB_DIR = os.path.join(DATA_DIR, "RGB_images")
MASK_DIR = os.path.join(DATA_DIR, "masks")

val_dataset = ResizedTreeSegmentationDataset(RGB_DIR, MASK_DIR, size=(360, 360))
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False)

# ===============================
# Load DeepLabV3 model with 4-channel input
# ===============================
model_name = "deeplabv3"
checkpoint_path = f"/content/COMP9517/checkpoints/{model_name}_best.pth"
base_dir = f"/content/COMP9517/outputs/{model_name}"
pred_dir = os.path.join(base_dir, "predictions")
npy_dir = os.path.join(pred_dir, "npy")
worst_dir = os.path.join(pred_dir, "worst")
conf_matrix_dir = os.path.join(base_dir, "confusion_matrix")
os.makedirs(npy_dir, exist_ok=True)
os.makedirs(worst_dir, exist_ok=True)
os.makedirs(conf_matrix_dir, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load pre-trained DeepLabV3 with ResNet-50 backbone
weights = models.segmentation.DeepLabV3_ResNet50_Weights.DEFAULT
model = models.segmentation.deeplabv3_resnet50(weights=weights)
model.classifier[4] = nn.Conv2d(256, 2, kernel_size=1)  # Two classes: background & tree

# Adapt first convolution layer for 4-channel input
state_dict = torch.load(checkpoint_path, map_location=device)
if "backbone.conv1.weight" in state_dict:
    del state_dict["backbone.conv1.weight"]
old_conv = model.backbone.conv1
new_conv = nn.Conv2d(4, old_conv.out_channels,
                     kernel_size=old_conv.kernel_size,
                     stride=old_conv.stride,
                     padding=old_conv.padding,
                     bias=old_conv.bias is not None)
with torch.no_grad():
    new_conv.weight[:, :3, :, :] = old_conv.weight  # Copy RGB weights
    new_conv.weight[:, 3:4, :, :] = old_conv.weight.mean(dim=1, keepdim=True)
    if old_conv.bias is not None:
        new_conv.bias = old_conv.bias
model.backbone.conv1 = new_conv

model.load_state_dict(state_dict, strict=False)
model.to(device)
model.eval()

# ===============================
# Evaluation loop
# ===============================
start_time = time.time()
total_acc = total_dice = total_iou = total_precision = total_recall = 0
num_samples = 0
errors = []
all_preds = []
all_targets = []
sample_paths = []

for idx, (image, mask) in enumerate(tqdm(val_loader, desc="DeepLabV3 Evaluation")):
    image = image.to(device)
    mask = mask.to(device)

    # Forward pass
    with torch.no_grad():
        pred = model(image)["out"]
        pred_bin = torch.argmax(pred, dim=1)  # Convert logits to class index

    # Compute metrics
    acc = accuracy_score(mask.view(-1).cpu().numpy(), pred_bin.view(-1).cpu().numpy())
    dice = f1_score(mask.view(-1).cpu().numpy(), pred_bin.view(-1).cpu().numpy(), zero_division=1)
    iou = jaccard_score(mask.view(-1).cpu().numpy(), pred_bin.view(-1).cpu().numpy(), zero_division=1)
    prec = precision_score(mask.view(-1).cpu().numpy(), pred_bin.view(-1).cpu().numpy(), zero_division=1)
    rec = recall_score(mask.view(-1).cpu().numpy(), pred_bin.view(-1).cpu().numpy(), zero_division=1)

    total_acc += acc
    total_dice += dice
    total_iou += iou
    total_precision += prec
    total_recall += rec
    num_samples += 1
    errors.append((idx, 1 - acc))   # Store error rate for worst-case selection

    # Save predictions and masks as .npy for further analysis
    np.save(os.path.join(npy_dir, f"pred_{idx}.npy"), pred_bin[0].cpu().numpy())
    np.save(os.path.join(npy_dir, f"mask_{idx}.npy"), mask[0].cpu().numpy())

    all_preds.append(pred_bin.view(-1).cpu().numpy())
    all_targets.append(mask.view(-1).cpu().numpy())
    sample_paths.append((image[0, :3].cpu(), mask[0].cpu(), pred_bin[0].cpu()))

# ===============================
# Summary statistics
# ===============================
avg_acc = total_acc / num_samples
avg_dice = total_dice / num_samples
avg_iou = total_iou / num_samples
avg_prec = total_precision / num_samples
avg_recall = total_recall / num_samples
total_time = time.time() - start_time

print(f"Total samples predicted: {num_samples}")
print(f"Average pixel-wise accuracy on validation set: {avg_acc:.4f}")
print(f"Average Dice: {avg_dice:.4f}")
print(f"Average IoU: {avg_iou:.4f}")
print(f"Average Precision: {avg_prec:.4f}")
print(f"Average Recall: {avg_recall:.4f}")
print(f"Total inference time: {total_time:.2f} seconds")
print(f"Prediction images saved in: {pred_dir}")

# Save metrics to CSV
eval_df = pd.DataFrame([{
    "Model": model_name.upper(),
    "Accuracy": avg_acc,
    "Dice": avg_dice,
    "IoU": avg_iou,
    "Precision": avg_prec,
    "Recall": avg_recall,
    "InferenceTimeSec": total_time
}])
eval_df.to_csv(os.path.join(base_dir, f"eval_results_{model_name}.csv"), index=False)

# ===============================
# Confusion Matrix generation
# ===============================
all_preds_np = np.concatenate(all_preds)
all_targets_np = np.concatenate(all_targets)
cm = confusion_matrix(all_targets_np, all_preds_np)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Background", "Tree"])
disp.plot(cmap=plt.cm.Blues, values_format="d")
plt.title(f"Confusion Matrix - {model_name.upper()}", fontsize=14, pad=10)
plt.grid(False)
plt.subplots_adjust(left=0.25)
confusion_image_path = os.path.join(conf_matrix_dir, f"{model_name}_confusion_matrix.png")
plt.savefig(confusion_image_path, bbox_inches='tight')
plt.close()
print(f"Confusion matrix saved to {confusion_image_path}")
display(Image(filename=confusion_image_path))

# ===============================
# Worst-case predictions
# ===============================
worst_k = 5
print("\nWorst-case prediction samples (highest prediction error):")
worst_samples = sorted(errors, key=lambda x: x[1], reverse=True)[:worst_k]
for idx, err in worst_samples:
    img_tensor, gt_mask_tensor, pred_tensor = sample_paths[idx]
    rgb = img_tensor.permute(1, 2, 0).numpy()
    rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min())
    gt_mask = gt_mask_tensor.squeeze().numpy()
    pred_mask = pred_tensor.squeeze().numpy()

    fig, axs = plt.subplots(1, 3, figsize=(15, 5))
    axs[0].imshow(rgb)
    axs[0].set_title("RGB Image", pad=10)
    axs[1].imshow(gt_mask, cmap="gray")
    axs[1].set_title("Ground Truth Mask", pad=10)
    axs[2].imshow(pred_mask, cmap="gray")
    axs[2].set_title("Predicted Mask", pad=10)
    plt.tight_layout()
    path = os.path.join(worst_dir, f"worst_{idx}.png")
    plt.savefig(path)
    plt.close()
    display(Image(filename=path))

# ===============================
# Random prediction samples
# ===============================
print("\nRandom prediction samples:")
random_indices = random.sample(range(len(sample_paths)), min(5, len(sample_paths)))
for idx in random_indices:
    img_tensor, gt_mask_tensor, pred_tensor = sample_paths[idx]
    rgb = img_tensor.permute(1, 2, 0).numpy()
    rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min())
    gt_mask = gt_mask_tensor.squeeze().numpy()
    pred_mask = pred_tensor.squeeze().numpy()

    fig, axs = plt.subplots(1, 3, figsize=(15, 5))
    axs[0].imshow(rgb)
    axs[0].set_title("RGB Image", pad=10)
    axs[1].imshow(gt_mask, cmap="gray")
    axs[1].set_title("Ground Truth Mask", pad=10)
    axs[2].imshow(pred_mask, cmap="gray")
    axs[2].set_title("Predicted Mask", pad=10)
    plt.tight_layout()
    plt.show()

## Model 4: Fast-SCNN Segmentation

Fast-SCNN is evaluated as a lightweight semantic segmentation model designed for efficient inference under resource constraints.  
Compared to heavier architectures, Fast-SCNN emphasizes computational efficiency while maintaining reasonable segmentation accuracy, making it suitable for large-scale or real-time aerial analysis scenarios.

The model operates on RGB aerial imagery and predicts pixel-level binary masks for standing dead tree segmentation.  
To mitigate the impact of class imbalance between background and tree pixels, a focal loss–based optimization strategy is employed, encouraging the model to focus on harder foreground examples.

Performance is evaluated using pixel accuracy, Dice score, and Intersection-over-Union (IoU), enabling direct comparison with higher-capacity models such as UNet and DeepLabV3.  
This analysis highlights the trade-offs between segmentation quality and computational efficiency across different model families.

In [ ]:
import os
import time
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from PIL import Image
from tqdm import tqdm
import pandas as pd
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader

# =========================
# Paths and configuration
# =========================
base_dir = "/content/COMP9517/USA_segmentation"
model_name = "fastscnn"
checkpoint_path = f"/content/COMP9517/checkpoints/{model_name}_best.pth"
log_path = f"/content/COMP9517/training_log_{model_name}.csv"
os.makedirs(os.path.dirname(checkpoint_path), exist_ok=True)

# =========================
# Custom Dataset Definition
# =========================
class CustomSegmentationDataset(Dataset):
    def __init__(self, rgb_dir, mask_dir, file_list, transform=None, target_size=(256, 256)):
        self.rgb_dir = rgb_dir
        self.mask_dir = mask_dir
        self.file_list = file_list
        self.transform = transform
        self.target_size = target_size

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        base = self.file_list[idx]
        rgb_path = os.path.join(self.rgb_dir, base + ".png")
        mask_path = os.path.join(self.mask_dir, "mask_" + base.split("RGB_")[-1] + ".png")

        # Load and resize RGB image
        image = Image.open(rgb_path).convert("RGB").resize(self.target_size, Image.BILINEAR)

        # Load and resize mask (nearest neighbor to preserve labels)
        mask = Image.open(mask_path).convert("L").resize(self.target_size, Image.NEAREST)

        # Apply transformations
        if self.transform:
            image = self.transform(image)
        mask = transforms.ToTensor()(mask).long().squeeze(0)    # Convert to Long tensor for CE loss

        return image, mask

# =========================
# FastSCNN Model Definition
# =========================
class FastSCNN(nn.Module):
    def __init__(self, n_classes):
        super(FastSCNN, self).__init__()
        self.downsample = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 48, 3, stride=2, padding=1),
            nn.BatchNorm2d(48), nn.ReLU(),
            nn.Conv2d(48, 64, 3, stride=2, padding=1),
            nn.BatchNorm2d(64), nn.ReLU()
        )
        self.classifier = nn.Sequential(
            nn.Conv2d(64, 64, 1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, n_classes, 1)
        )

    def forward(self, x):
        x = self.downsample(x)
        x = self.classifier(x)

        # Upsample predictions back to input size
        return nn.functional.interpolate(x, scale_factor=8, mode='bilinear', align_corners=True)

# =========================
# Focal Loss Implementation
# =========================
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce_loss = nn.functional.cross_entropy(inputs, targets, weight=self.alpha, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean()

# =========================
# Training Configuration
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

input_size = (256, 256)
batch_size = 8
num_epochs = 20
num_classes = 2
lr = 0.001

# =========================
# Dataset Preparation
# =========================
all_files = [f[:-4] for f in os.listdir(os.path.join(base_dir, "RGB_images")) if f.endswith(".png")]
random.shuffle(all_files)
split = int(0.8 * len(all_files))
train_files = all_files[:split]
val_files = all_files[split:]

transform = transforms.ToTensor()
train_dataset = CustomSegmentationDataset(os.path.join(base_dir, "RGB_images"), os.path.join(base_dir, "masks"), train_files, transform, input_size)
val_dataset = CustomSegmentationDataset(os.path.join(base_dir, "RGB_images"), os.path.join(base_dir, "masks"), val_files, transform, input_size)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

# =========================
# Model, Loss, Optimizer
# =========================
model = FastSCNN(n_classes=num_classes).to(device)
class_weights = torch.tensor([0.1, 0.9], dtype=torch.float32).to(device)
criterion = FocalLoss(alpha=class_weights)
optimizer = optim.Adam(model.parameters(), lr=lr)

# =========================
# Training Loop
# =========================
best_dice = 0.0
best_iou = 0.0
best_acc = 0.0
best_dice_epoch = 0
best_iou_epoch = 0
best_acc_epoch = 0
log = []

for epoch in range(1, num_epochs + 1):
    start_time = time.time()
    model.train()
    train_loss = 0
    train_bar = tqdm(train_loader, desc=f"Epoch {epoch}/{num_epochs}")
    for images, masks in train_bar:
        images, masks = images.to(device), masks.to(device)
        outputs = model(images)
        loss = criterion(outputs, masks)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        train_bar.set_postfix(loss=loss.item())

    avg_train_loss = train_loss / len(train_loader)

    # Validation loop
    model.eval()
    val_loss = 0
    total_correct = 0
    total_pixels = 0
    total_iou = 0
    total_dice = 0
    val_bar = tqdm(val_loader, desc="Validation")
    with torch.no_grad():
        for images, masks in val_bar:
            images, masks = images.to(device), masks.to(device)
            outputs = model(images)
            loss = criterion(outputs, masks)
            val_loss += loss.item()

            preds = torch.argmax(outputs, dim=1)
            correct = (preds == masks).sum().item()
            total_correct += correct
            total_pixels += torch.numel(preds)

            intersection = ((preds == 1) & (masks == 1)).sum().item()
            union = ((preds == 1) | (masks == 1)).sum().item()
            iou = intersection / union if union > 0 else 0
            dice = (2 * intersection) / (preds.eq(1).sum().item() + masks.eq(1).sum().item() + 1e-8)
            total_iou += iou
            total_dice += dice

    avg_val_loss = val_loss / len(val_loader)
    avg_acc = total_correct / total_pixels
    avg_iou = total_iou / len(val_loader)
    avg_dice = total_dice / len(val_loader)
    elapsed = time.time() - start_time

    # Log and checkpoint saving
    print()
    print(f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | IoU: {avg_iou:.4f} | Dice: {avg_dice:.4f} | Accuracy: {avg_acc:.4f} | Time: {elapsed:.1f}s")

    if avg_dice > best_dice:
        best_dice = avg_dice
        best_dice_epoch = epoch
        torch.save(model.state_dict(), checkpoint_path)
        print("Saved best model (based on Dice).\n")
    else:
        print("No improvement in Dice.\n")

    if avg_iou > best_iou:
        best_iou = avg_iou
        best_iou_epoch = epoch
    if avg_acc > best_acc:
        best_acc = avg_acc
        best_acc_epoch = epoch

    log.append({
        "Epoch": epoch,
        "Train Loss": avg_train_loss,
        "Val Loss": avg_val_loss,
        "IoU": avg_iou,
        "Dice": avg_dice,
        "Accuracy": avg_acc,
        "Time": elapsed
    })

# =========================
# Save training log
# =========================
total_time = sum(entry["Time"] for entry in log)
log_df = pd.DataFrame(log)
log_df.to_csv(log_path, index=False)

print()
print(f"Training finished. Log saved to {log_path}")
print(f"Best validation Dice: {best_dice:.4f} (Epoch {best_dice_epoch})")
print(f"Best validation Accuracy: {best_acc:.4f} (Epoch {best_acc_epoch})")
print(f"Best validation IoU: {best_iou:.4f} (Epoch {best_iou_epoch})")
print(f"Total training time: {total_time:.1f} seconds")

### Fast-SCNN Training Dynamics and Stability Analysis

Training dynamics of the Fast-SCNN model are analyzed through visualization of loss and segmentation performance metrics across epochs.  
Key indicators such as training and validation loss, Dice score, Intersection-over-Union (IoU), and pixel accuracy are monitored to assess convergence behavior and optimization stability.

These training curves provide insight into how effectively the lightweight architecture learns under class imbalance and facilitate direct comparison with higher-capacity models such as UNet and DeepLabV3.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os
from IPython.display import display, Image

def visualize_training_log(log_path="/content/COMP9517/training_log_fastscnn.csv"):
    # Load the training log CSV into a pandas DataFrame
    log = pd.read_csv(log_path)

    # Ensure output directory exists for saving plots
    output_dir = "/content/COMP9517/outputs/fastscnn/plots"
    os.makedirs(output_dir, exist_ok=True)

    # If Dice values are stored as tensor strings, extract numeric values
    if log["Dice"].dtype == object and log["Dice"].astype(str).str.contains("tensor").any():
        log["Dice"] = log["Dice"].apply(lambda x: float(str(x).split("(")[-1].split(",")[0]))

    # Helper function to generate and save a single metric plot
    def save_plot(x, y, ylabel, title, filename, color=None):
        plt.figure()
        plt.plot(x, y, label=ylabel, color=color)
        plt.xlabel("Epoch")
        plt.ylabel(ylabel)
        plt.title(title)
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, filename), bbox_inches='tight')
        plt.close()

    # Generate and save plots for each metric
    save_plot(log["Epoch"], log["Train Loss"],
              "Train Loss", "Train Loss over Epochs", "train_loss_curve.png")
    save_plot(log["Epoch"], log["Val Loss"],
              "Validation Loss", "Validation Loss over Epochs", "val_loss_curve.png")
    save_plot(log["Epoch"], log["Accuracy"],
              "Accuracy", "Validation Accuracy over Epochs", "accuracy_curve.png", "green")
    save_plot(log["Epoch"], log["Dice"],
              "Dice", "Validation Dice over Epochs", "dice_curve.png", "purple")
    save_plot(log["Epoch"], log["IoU"],
              "IoU", "Validation IoU over Epochs", "iou_curve.png", "orange")

    print("FastSCNN training plots saved to:", output_dir)

    # Optionally preview plots in the notebook
    print("\n[Preview of Training Plots]")
    for name in ["train_loss_curve.png", "val_loss_curve.png", "accuracy_curve.png", "dice_curve.png", "iou_curve.png"]:
        path = os.path.join(output_dir, name)
        if os.path.exists(path):
            print(f"Showing: {name}")
            display(Image(filename=path))

# Execute visualization only if run as a script
if __name__ == "__main__":
    visualize_training_log("/content/COMP9517/training_log_fastscnn.csv")

### Fast-SCNN Validation Performance Analysis

The trained Fast-SCNN model is evaluated on a held-out validation set to assess segmentation quality under a lightweight architecture.  
Performance is measured using complementary metrics including pixel accuracy, Dice score, Intersection-over-Union (IoU), precision, and recall, providing a balanced view of pixel-level and region-level behavior.

In addition to aggregated metrics, qualitative analysis is conducted through inspection of representative predictions and failure cases.  
Visualization of both typical and challenging samples highlights the strengths and limitations of Fast-SCNN in complex aerial scenes and enables direct comparison with higher-capacity models such as UNet and DeepLabV3.

In [ ]:
import os
import time
import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    precision_score, recall_score, f1_score, jaccard_score, accuracy_score
)
from IPython.display import display, Image
import random

from torch.utils.data import Dataset, DataLoader
from PIL import Image as PILImage
from torchvision.transforms import functional as TF

# ===============================
# Dataset class for RGB input images and binary masks
# ===============================
class ResizedTreeSegmentationDataset(Dataset):
    def __init__(self, rgb_dir, mask_dir, size=(256, 256)):
        self.rgb_dir = rgb_dir
        self.mask_dir = mask_dir
        self.filenames = sorted([f for f in os.listdir(rgb_dir) if f.endswith('.png')])
        self.size = size

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        fname = self.filenames[idx]
        rgb_path = os.path.join(self.rgb_dir, fname)
        mask_path = os.path.join(self.mask_dir, "mask_" + fname.replace("RGB_", ""))

        # Load and convert RGB image and mask
        rgb = PILImage.open(rgb_path).convert("RGB")
        mask = PILImage.open(mask_path).convert("L")

        # Resize both image and mask
        rgb = TF.resize(rgb, self.size)
        mask = TF.resize(mask, self.size, interpolation=PILImage.NEAREST)

        # Convert to tensor
        rgb = TF.to_tensor(rgb)
        mask = np.array(mask)
        mask = (mask > 127).astype(np.uint8)    # Binary mask
        mask = torch.tensor(mask, dtype=torch.long)

        return rgb, mask

# ===============================
# Minimal FastSCNN model definition
# ===============================
from torch import nn

class FastSCNN(nn.Module):
    def __init__(self, n_classes):
        super(FastSCNN, self).__init__()
        self.downsample = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 48, 3, stride=2, padding=1),
            nn.BatchNorm2d(48), nn.ReLU(),
            nn.Conv2d(48, 64, 3, stride=2, padding=1),
            nn.BatchNorm2d(64), nn.ReLU()
        )
        self.classifier = nn.Sequential(
            nn.Conv2d(64, 64, 1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, n_classes, 1)
        )

    def forward(self, x):
        x = self.downsample(x)
        x = self.classifier(x)
        # Upsample to original resolution
        return nn.functional.interpolate(x, scale_factor=8, mode='bilinear', align_corners=True)

# ===============================
# Paths and setup
# ===============================
model_name = "fastscnn"
checkpoint_path = f"/content/COMP9517/checkpoints/{model_name}_best.pth"
DATA_DIR = "/content/COMP9517/USA_segmentation"
RGB_DIR = os.path.join(DATA_DIR, "RGB_images")
MASK_DIR = os.path.join(DATA_DIR, "masks")

# Output directories
base_dir = f"/content/COMP9517/outputs/{model_name}"
pred_dir = os.path.join(base_dir, "predictions")
npy_dir = os.path.join(pred_dir, "npy")
worst_dir = os.path.join(pred_dir, "worst")
conf_matrix_dir = os.path.join(base_dir, "confusion_matrix")
os.makedirs(npy_dir, exist_ok=True)
os.makedirs(worst_dir, exist_ok=True)
os.makedirs(conf_matrix_dir, exist_ok=True)

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load model and weights
model = FastSCNN(n_classes=2)
model.load_state_dict(torch.load(checkpoint_path, map_location=device))
model.to(device)
model.eval()

# Load validation dataset
val_dataset = ResizedTreeSegmentationDataset(RGB_DIR, MASK_DIR, size=(256, 256))
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False)

# ===============================
# Evaluation loop
# ===============================
start_time = time.time()
total_acc = total_dice = total_iou = total_precision = total_recall = 0
num_samples = 0
errors = []         # List of (index, error_rate) for worst-case selection
all_preds = []        # Flattened predictions
all_targets = []      # Flattened ground truth masks
sample_paths = []     # (image_tensor, gt_mask, pred_mask) for visualization

for idx, (image, mask) in enumerate(tqdm(val_loader, desc="FastSCNN Evaluation")):
    image = image.to(device)
    mask = mask.to(device)

    with torch.no_grad():
        pred = model(image)
        pred_bin = torch.argmax(pred, dim=1)

    # Compute metrics for current sample
    acc = accuracy_score(mask.view(-1).cpu().numpy(), pred_bin.view(-1).cpu().numpy())
    dice = f1_score(mask.view(-1).cpu().numpy(), pred_bin.view(-1).cpu().numpy(), zero_division=1)
    iou = jaccard_score(mask.view(-1).cpu().numpy(), pred_bin.view(-1).cpu().numpy(), zero_division=1)
    prec = precision_score(mask.view(-1).cpu().numpy(), pred_bin.view(-1).cpu().numpy(), zero_division=1)
    rec = recall_score(mask.view(-1).cpu().numpy(), pred_bin.view(-1).cpu().numpy(), zero_division=1)

    # Accumulate totals
    total_acc += acc
    total_dice += dice
    total_iou += iou
    total_precision += prec
    total_recall += rec
    num_samples += 1
    errors.append((idx, 1 - acc))

    # Save predictions and masks as .npy
    np.save(os.path.join(npy_dir, f"pred_{idx}.npy"), pred_bin[0].cpu().numpy())
    np.save(os.path.join(npy_dir, f"mask_{idx}.npy"), mask[0].cpu().numpy())

    # Store for confusion matrix and visualization
    all_preds.append(pred_bin.view(-1).cpu().numpy())
    all_targets.append(mask.view(-1).cpu().numpy())
    sample_paths.append((image[0].cpu(), mask[0].cpu(), pred_bin[0].cpu()))

# ===============================
# Summary statistics
# ===============================
avg_acc = total_acc / num_samples
avg_dice = total_dice / num_samples
avg_iou = total_iou / num_samples
avg_prec = total_precision / num_samples
avg_recall = total_recall / num_samples
total_time = time.time() - start_time

print(f"Total samples predicted: {num_samples}")
print(f"Average pixel-wise accuracy on validation set: {avg_acc:.4f}")
print(f"Average Dice: {avg_dice:.4f}")
print(f"Average IoU: {avg_iou:.4f}")
print(f"Average Precision: {avg_prec:.4f}")
print(f"Average Recall: {avg_recall:.4f}")
print(f"Total inference time: {total_time:.2f} seconds")
print(f"Prediction images saved in: {pred_dir}")

# Save evaluation results to CSV
eval_df = pd.DataFrame([{
    "Model": model_name.upper(),
    "Accuracy": avg_acc,
    "Dice": avg_dice,
    "IoU": avg_iou,
    "Precision": avg_prec,
    "Recall": avg_recall,
    "InferenceTimeSec": total_time
}])
eval_df.to_csv(os.path.join(base_dir, f"eval_results_{model_name}.csv"), index=False)

# ===============================
# Confusion Matrix
# ===============================
all_preds_np = np.concatenate(all_preds)
all_targets_np = np.concatenate(all_targets)
cm = confusion_matrix(all_targets_np, all_preds_np)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Background", "Tree"])
disp.plot(cmap=plt.cm.Blues, values_format="d")
plt.title(f"Confusion Matrix - {model_name.upper()}", fontsize=14, pad=10)
plt.grid(False)
plt.subplots_adjust(left=0.25)
confusion_image_path = os.path.join(conf_matrix_dir, f"{model_name}_confusion_matrix.png")
plt.savefig(confusion_image_path, bbox_inches='tight')
plt.close()
print(f"Confusion matrix saved to {confusion_image_path}")
display(Image(filename=confusion_image_path))

# ===============================
# Worst-case prediction samples
# ===============================
worst_k = 5
print("\nWorst-case prediction samples (highest prediction error):")
worst_samples = sorted(errors, key=lambda x: x[1], reverse=True)[:worst_k]
for idx, err in worst_samples:
    img_tensor, gt_mask_tensor, pred_tensor = sample_paths[idx]
    rgb = img_tensor.permute(1, 2, 0).numpy()
    rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min())
    gt_mask = gt_mask_tensor.squeeze().numpy()
    pred_mask = pred_tensor.squeeze().numpy()

    fig, axs = plt.subplots(1, 3, figsize=(15, 5))
    axs[0].imshow(rgb)
    axs[0].set_title("RGB Image", pad=10)
    axs[1].imshow(gt_mask, cmap="gray")
    axs[1].set_title("Ground Truth Mask", pad=10)
    axs[2].imshow(pred_mask, cmap="gray")
    axs[2].set_title("Predicted Mask", pad=10)
    plt.tight_layout()
    path = os.path.join(worst_dir, f"worst_{idx}.png")
    plt.savefig(path)
    plt.close()
    display(Image(filename=path))

# ===============================
# Random prediction samples
# ===============================
print("\nRandom prediction samples:")
random_indices = random.sample(range(len(sample_paths)), min(5, len(sample_paths)))
for idx in random_indices:
    img_tensor, gt_mask_tensor, pred_tensor = sample_paths[idx]
    rgb = img_tensor.permute(1, 2, 0).numpy()
    rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min())
    gt_mask = gt_mask_tensor.squeeze().numpy()
    pred_mask = pred_tensor.squeeze().numpy()

    fig, axs = plt.subplots(1, 3, figsize=(15, 5))
    axs[0].imshow(rgb)
    axs[0].set_title("RGB Image", pad=10)
    axs[1].imshow(gt_mask, cmap="gray")
    axs[1].set_title("Ground Truth Mask", pad=10)
    axs[2].imshow(pred_mask, cmap="gray")
    axs[2].set_title("Predicted Mask", pad=10)
    plt.tight_layout()
    plt.show()

## Model 5: ADA-Net Segmentation

ADA-Net is evaluated as a custom lightweight encoder–decoder segmentation model designed for efficient binary segmentation of standing dead trees in aerial imagery.  
The architecture follows a compact design that progressively encodes spatial features and restores resolution through a symmetric decoding pathway, aiming to balance segmentation accuracy and computational efficiency.

The model is trained using a supervised segmentation pipeline and optimized with a standard classification loss for pixel-level prediction.  
Performance is assessed using pixel accuracy, Dice score, and Intersection-over-Union (IoU), enabling direct comparison with other lightweight and high-capacity segmentation models evaluated in this project.

ADA-Net provides an additional perspective on efficiency-oriented model design and serves as a useful comparison point against Fast-SCNN, UNet, and DeepLabV3 within the overall segmentation study.

In [ ]:
import os
import time
import torch
import torch.nn as nn
import numpy as np
from PIL import Image
from tqdm import tqdm
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader

# ===============================
# Custom dataset for aerial tree segmentation
# ===============================
class CustomSegmentationDataset(Dataset):
    def __init__(self, root_dir, split="train", transform=None):
        self.image_dir = os.path.join(root_dir, "RGB_images")
        self.mask_dir = os.path.join(root_dir, "masks")
        self.transform = transform

        # Load all image filenames and split into train/val sets
        file_list = sorted([f for f in os.listdir(self.image_dir) if f.endswith(".png")])
        split_idx = int(0.8 * len(file_list))
        self.file_list = file_list[:split_idx] if split == "train" else file_list[split_idx:]

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        img_name = self.file_list[idx]
        rgb_path = os.path.join(self.image_dir, img_name)
        mask_path = os.path.join(self.mask_dir, img_name.replace("RGB_", "mask_"))

        # Load image and mask
        image = Image.open(rgb_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")

        if self.transform:
            # Apply transforms to image
            image = self.transform(image)
            # Resize mask using nearest neighbor to preserve class labels
            mask = transforms.Resize((256, 256), interpolation=Image.NEAREST)(mask)
            mask = torch.from_numpy(np.array(mask)).long()
            mask = (mask > 127).long()  # Convert to binary mask

        return image, mask

# ===============================
# Minimal ADA-Net architecture
# ===============================
class AdaNet(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 128, 3, stride=2, padding=1), nn.BatchNorm2d(128), nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 2, stride=2), nn.BatchNorm2d(64), nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 2, stride=2), nn.BatchNorm2d(32), nn.ReLU(),
            nn.ConvTranspose2d(32, n_classes, 2, stride=2)
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))

# ===============================
# Configuration
# ===============================
data_dir = "/content/COMP9517/USA_segmentation"
model_path = "/content/COMP9517/checkpoints/adanet_best.pth"
log_path = "/content/COMP9517/training_log_adanet.csv"
os.makedirs(os.path.dirname(model_path), exist_ok=True)

batch_size = 8
learning_rate = 0.001
num_epochs = 20
num_classes = 2

# Image transformations
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor()
])

# Data loaders for train and validation sets
train_loader = DataLoader(CustomSegmentationDataset(data_dir, "train", transform),
                          batch_size=batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(CustomSegmentationDataset(data_dir, "val", transform),
                        batch_size=1, shuffle=False, num_workers=2)

# ===============================
# Device setup
# ===============================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Model, loss, optimizer
model = AdaNet(num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

# ===============================
# Training loop
# ===============================
best_dice = 0.0
best_iou = 0.0
best_acc = 0.0
best_dice_epoch = 0
best_iou_epoch = 0
best_acc_epoch = 0
log_lines = ["epoch,train_loss,val_loss,iou,dice,acc"]
start_time = time.time()

for epoch in range(num_epochs):
    # ---- Training phase ----
    model.train()
    total_loss = 0.0
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")

    for images, masks in loop:
        images, masks = images.to(device), masks.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        loop.set_postfix(loss=loss.item())

    train_loss = total_loss / len(train_loader)

    # ---- Validation phase ----
    model.eval()
    val_loss, total_iou, total_dice, total_acc = 0.0, 0.0, 0.0, 0.0

    with torch.no_grad():
        for images, masks in tqdm(val_loader, desc="Validation"):
            images, masks = images.to(device), masks.to(device)
            outputs = model(images)
            loss = criterion(outputs, masks)
            val_loss += loss.item()

            preds = torch.argmax(outputs, dim=1)

            # Compute intersection, union, Dice, and accuracy
            intersection = torch.logical_and(preds == 1, masks == 1).sum().item()
            union = torch.logical_or(preds == 1, masks == 1).sum().item()
            dice = (2 * intersection) / (preds.eq(1).sum().item() + masks.eq(1).sum().item() + 1e-8)
            acc = (preds == masks).float().mean().item()

            total_iou += intersection / (union + 1e-8)
            total_dice += dice
            total_acc += acc

    val_loss /= len(val_loader)
    avg_iou = total_iou / len(val_loader)
    avg_dice = total_dice / len(val_loader)
    avg_acc = total_acc / len(val_loader)

    # ---- Logging ----
    print()
    print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | IoU: {avg_iou:.4f} | Dice: {avg_dice:.4f} | Accuracy: {avg_acc:.4f}")

    # Save best model based on Dice score
    if avg_dice > best_dice:
        best_dice = avg_dice
        best_dice_epoch = epoch + 1
        torch.save(model.state_dict(), model_path)
        print("Saved best model (based on Dice).\n")
    else:
        print("No improvement in Dice.\n")

    if avg_iou > best_iou:
        best_iou = avg_iou
        best_iou_epoch = epoch + 1

    if avg_acc > best_acc:
        best_acc = avg_acc
        best_acc_epoch = epoch + 1

    log_lines.append(f"{epoch+1},{train_loss:.4f},{val_loss:.4f},{avg_iou:.4f},{avg_dice:.4f},{avg_acc:.4f}")

# ===============================
# Save training log
# ===============================
with open(log_path, "w") as f:
    f.write("\n".join(log_lines))

# ===============================
# Final summary
# ===============================
total_time = time.time() - start_time
print()
print(f"Training finished. Log saved to {log_path}")
print(f"Best validation Dice: {best_dice:.4f} (Epoch {best_dice_epoch})")
print(f"Best validation Accuracy: {best_acc:.4f} (Epoch {best_acc_epoch})")
print(f"Best validation IoU: {best_iou:.4f} (Epoch {best_iou_epoch})")
print(f"Total training time: {total_time:.1f} seconds")

### ADA-Net Training Dynamics and Stability Analysis

Training behavior of the ADA-Net model is analyzed through visualization of loss and segmentation performance metrics across epochs.  
Key indicators such as training and validation loss, Dice score, Intersection-over-Union (IoU), and pixel accuracy are monitored to assess convergence behavior and optimization stability.

These training dynamics provide insight into how effectively the lightweight encoder–decoder architecture learns under the given data distribution and enable direct comparison with other models such as UNet and Fast-SCNN.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os
from IPython.display import display, Image

def visualize_training_log(log_path="/content/COMP9517/training_log_adanet.csv"):
    # Load the training log CSV into a DataFrame
    log = pd.read_csv(log_path)

    # Ensure the output directory exists for saving plots
    output_dir = "/content/COMP9517/outputs/adanet/plots"
    os.makedirs(output_dir, exist_ok=True)

    # Helper function to generate and save a plot
    def save_plot(x, y, ylabel, title, filename, color=None):
        plt.figure()
        plt.plot(x, y, label=ylabel, color=color)
        plt.xlabel("Epoch")
        plt.ylabel(ylabel)
        plt.title(title)
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, filename), bbox_inches='tight')
        plt.close()

    # Generate and save plots for each key metric
    save_plot(log["epoch"], log["train_loss"],
              "Train Loss", "Train Loss over Epochs", "train_loss_curve.png")
    save_plot(log["epoch"], log["val_loss"],
              "Validation Loss", "Validation Loss over Epochs", "val_loss_curve.png")
    save_plot(log["epoch"], log["acc"],
              "Accuracy", "Validation Accuracy over Epochs", "accuracy_curve.png", "green")
    save_plot(log["epoch"], log["dice"],
              "Dice", "Validation Dice over Epochs", "dice_curve.png", "purple")
    save_plot(log["epoch"], log["iou"],
              "IoU", "Validation IoU over Epochs", "iou_curve.png", "orange")

    print("ADA-Net training plots saved to:", output_dir)

    # Display the generated plots inline (for Jupyter/Colab preview)
    print("\n[Preview of Training Plots]")
    for name in ["train_loss_curve.png", "val_loss_curve.png", "accuracy_curve.png", "dice_curve.png", "iou_curve.png"]:
        path = os.path.join(output_dir, name)
        if os.path.exists(path):
            print(f"Showing: {name}")
            display(Image(filename=path))

# Run the visualization when executing the script directly
if __name__ == "__main__":
    visualize_training_log("/content/COMP9517/training_log_adanet.csv")

### ADA-Net Validation Performance Analysis

The trained ADA-Net model is evaluated on a held-out validation set to assess its effectiveness for binary segmentation of standing dead trees.  
Performance is measured using complementary metrics including pixel accuracy, Dice score, Intersection-over-Union (IoU), precision, and recall, providing a comprehensive evaluation of segmentation quality.

Beyond aggregate metrics, qualitative analysis is conducted through inspection of representative predictions and failure cases.  
Visualization of typical and challenging samples highlights how the lightweight ADA-Net architecture handles complex aerial scenes and enables direct comparison with other models such as Fast-SCNN, DeepLabV3, and UNet.

In [ ]:
import os
import time
import torch
import numpy as np
import pandas as pd
import random
import matplotlib.pyplot as plt
from PIL import Image as PILImage
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    precision_score, recall_score, f1_score, jaccard_score, accuracy_score
)
from IPython.display import display, Image

# ===============================
# Setup paths and environment
# ===============================
model_name = "adanet"
checkpoint_path = f"/content/COMP9517/checkpoints/{model_name}_best.pth"
base_dir = f"/content/COMP9517/outputs/{model_name}"
pred_dir = os.path.join(base_dir, "predictions")
npy_dir = os.path.join(pred_dir, "npy")
worst_dir = os.path.join(pred_dir, "worst")
conf_matrix_dir = os.path.join(base_dir, "confusion_matrix")

# Create necessary directories
os.makedirs(npy_dir, exist_ok=True)
os.makedirs(worst_dir, exist_ok=True)
os.makedirs(conf_matrix_dir, exist_ok=True)

# Select computation device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ===============================
# Model definition and loading
# ===============================
class AdaNet(torch.nn.Module):
    def __init__(self, n_classes):
        super().__init__()

        # Encoder: downsampling feature extractor
        self.encoder = torch.nn.Sequential(
            torch.nn.Conv2d(3, 32, kernel_size=3, padding=1, stride=2),
            torch.nn.BatchNorm2d(32), torch.nn.ReLU(),
            torch.nn.Conv2d(32, 64, kernel_size=3, padding=1, stride=2),
            torch.nn.BatchNorm2d(64), torch.nn.ReLU(),
            torch.nn.Conv2d(64, 128, kernel_size=3, padding=1, stride=2),
            torch.nn.BatchNorm2d(128), torch.nn.ReLU()
        )

        # Decoder: upsampling to segmentation mask
        self.decoder = torch.nn.Sequential(
            torch.nn.ConvTranspose2d(128, 64, 2, stride=2),
            torch.nn.BatchNorm2d(64), torch.nn.ReLU(),
            torch.nn.ConvTranspose2d(64, 32, 2, stride=2),
            torch.nn.BatchNorm2d(32), torch.nn.ReLU(),
            torch.nn.ConvTranspose2d(32, n_classes, 2, stride=2)
        )

    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        return x

# Instantiate and load trained weights
model = AdaNet(n_classes=2)
model.load_state_dict(torch.load(checkpoint_path, map_location=device))
model.to(device)
model.eval()

# ===============================
# Dataset definition
# ===============================
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader

class AdanetDataset(Dataset):
    def __init__(self, rgb_dir, mask_dir):
        self.rgb_dir = rgb_dir
        self.mask_dir = mask_dir
        self.file_list = sorted([f for f in os.listdir(rgb_dir) if f.endswith(".png")])
        self.transform = transforms.Compose([
            transforms.Resize((256, 256)),
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        fname = self.file_list[idx]

        # Load RGB image
        rgb = PILImage.open(os.path.join(self.rgb_dir, fname)).convert("RGB")

        # Load corresponding binary mask
        mask = PILImage.open(os.path.join(self.mask_dir, fname.replace("RGB_", "mask_"))).convert("L")

        # Apply resizing and tensor conversion
        rgb = self.transform(rgb)
        mask = transforms.Resize((256, 256), interpolation=PILImage.NEAREST)(mask)
        mask = torch.tensor(np.array(mask), dtype=torch.long)
        mask = (mask > 127).long()
        return rgb, mask

# Prepare validation data loader
data_dir = "/content/COMP9517/USA_segmentation"
val_dataset = AdanetDataset(
    os.path.join(data_dir, "RGB_images"),
    os.path.join(data_dir, "masks")
)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False)

# ===============================
# Model evaluation
# ===============================
start_time = time.time()
total_acc = total_dice = total_iou = total_prec = total_recall = 0
num_samples = 0
errors = []            # Track per-sample errors
all_preds = []         # Flattened predictions for metrics
all_targets = []       # Flattened ground truths for metrics
sample_paths = []      # Store tensors for later visualization

for idx, (image, mask) in enumerate(tqdm(val_loader, desc="AdaNet Evaluation")):
    image = image.to(device)
    mask = mask.to(device)

    with torch.no_grad():
        pred = model(image)
        pred_bin = torch.argmax(pred, dim=1)

    # Compute evaluation metrics
    acc = accuracy_score(mask.view(-1).cpu(), pred_bin.view(-1).cpu())
    dice = f1_score(mask.view(-1).cpu(), pred_bin.view(-1).cpu(), zero_division=1)
    iou = jaccard_score(mask.view(-1).cpu(), pred_bin.view(-1).cpu(), zero_division=1)
    prec = precision_score(mask.view(-1).cpu(), pred_bin.view(-1).cpu(), zero_division=1)
    rec = recall_score(mask.view(-1).cpu(), pred_bin.view(-1).cpu(), zero_division=1)

    # Aggregate metrics
    total_acc += acc
    total_dice += dice
    total_iou += iou
    total_prec += prec
    total_recall += rec
    num_samples += 1
    errors.append((idx, 1 - acc))

    # Save predictions and masks as .npy for future use
    np.save(os.path.join(npy_dir, f"pred_{idx}.npy"), pred_bin[0].cpu().numpy())
    np.save(os.path.join(npy_dir, f"mask_{idx}.npy"), mask[0].cpu().numpy())

    all_preds.append(pred_bin.view(-1).cpu().numpy())
    all_targets.append(mask.view(-1).cpu().numpy())
    sample_paths.append((image[0].cpu(), mask[0].cpu(), pred_bin[0].cpu()))

# ===============================
# Summary statistics
# ===============================
avg_acc = total_acc / num_samples
avg_dice = total_dice / num_samples
avg_iou = total_iou / num_samples
avg_prec = total_prec / num_samples
avg_recall = total_recall / num_samples
elapsed = time.time() - start_time

print(f"Total samples predicted: {num_samples}")
print(f"Average pixel-wise accuracy on validation set: {avg_acc:.4f}")
print(f"Average Dice: {avg_dice:.4f}")
print(f"Average IoU: {avg_iou:.4f}")
print(f"Average Precision: {avg_prec:.4f}")
print(f"Average Recall: {avg_recall:.4f}")
print(f"Total inference time: {total_time:.2f} seconds")
print(f"Prediction images saved in: {pred_dir}")

# Save metrics to CSV
pd.DataFrame([{
    "Model": model_name.upper(), "Accuracy": avg_acc,
    "Dice": avg_dice, "IoU": avg_iou,
    "Precision": avg_prec, "Recall": avg_recall,
    "InferenceTimeSec": elapsed
}]).to_csv(os.path.join(base_dir, f"eval_results_{model_name}.csv"), index=False)

# ===============================
# Confusion matrix visualization
# ===============================
all_preds_np = np.concatenate(all_preds)
all_targets_np = np.concatenate(all_targets)
cm = confusion_matrix(all_targets_np, all_preds_np)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Background", "Tree"])
disp.plot(cmap=plt.cm.Blues, values_format="d")
plt.title(f"Confusion Matrix - {model_name.upper()}")
plt.grid(False)
plt.subplots_adjust(left=0.25)
conf_path = os.path.join(conf_matrix_dir, f"{model_name}_confusion_matrix.png")
plt.savefig(conf_path, bbox_inches='tight')
plt.close()
print(f"Confusion matrix saved to {conf_path}")
display(Image(filename=conf_path))

# ===============================
# Worst-case prediction samples
# ===============================
print("\nWorst-case prediction samples:")
for idx, _ in sorted(errors, key=lambda x: x[1], reverse=True)[:5]:
    rgb, gt, pr = sample_paths[idx]
    fig, axs = plt.subplots(1, 3, figsize=(15, 5))
    axs[0].imshow(rgb.permute(1, 2, 0))
    axs[0].set_title("RGB")
    axs[1].imshow(gt, cmap="gray")
    axs[1].set_title("Ground Truth")
    axs[2].imshow(pr, cmap="gray")
    axs[2].set_title("Prediction")
    plt.tight_layout()
    path = os.path.join(worst_dir, f"worst_{idx}.png")
    plt.savefig(path)
    plt.close()
    display(Image(filename=path))

# ===============================
# Random prediction samples
# ===============================
print("\nRandom prediction samples:")
for idx in random.sample(range(len(sample_paths)), min(5, len(sample_paths))):
    rgb, gt, pr = sample_paths[idx]
    fig, axs = plt.subplots(1, 3, figsize=(15, 5))
    axs[0].imshow(rgb.permute(1, 2, 0))
    axs[0].set_title("RGB")
    axs[1].imshow(gt, cmap="gray")
    axs[1].set_title("Ground Truth")
    axs[2].imshow(pr, cmap="gray")
    axs[2].set_title("Prediction")
    plt.tight_layout()
    plt.show()

## Summary of Individual Model Evaluation

This project implements and evaluates five segmentation approaches for identifying standing dead trees in aerial forest imagery, covering a broad spectrum of model complexity and design philosophy:

1. **SVM** – A traditional machine learning baseline based on handcrafted features.
2. **UNet** – A convolutional encoder–decoder architecture widely used for fine-grained segmentation.
3. **DeepLabV3 (ResNet-50 backbone)** – A high-capacity semantic segmentation model leveraging atrous convolution and multi-scale contextual features.
4. **Fast-SCNN** – A lightweight network designed for efficient and real-time segmentation.
5. **ADA-Net** – A custom encoder–decoder model optimized for efficiency in aerial imagery segmentation.

All models are trained and evaluated under a unified experimental setup using the same dataset split and consistent evaluation metrics, including **pixel accuracy**, **Dice score**, **Intersection-over-Union (IoU)**, **precision**, and **recall**.  
Both quantitative metrics and qualitative visual analyses are used to assess segmentation quality and failure modes across models.

With individual model behavior established, the following section presents a comparative analysis to examine performance trade-offs, robustness, and practical suitability across different segmentation approaches.

# Comparative Analysis of Segmentation Models

## Quantitative Performance Comparison

To systematically compare the effectiveness of different segmentation approaches, the final performance of all five models — **SVM**, **UNet**, **DeepLabV3**, **Fast-SCNN**, and **ADA-Net** — is analyzed using a consistent set of evaluation metrics.

The comparison focuses on the following indicators:
- **Intersection-over-Union (IoU)**: Measures region-level overlap between predictions and ground truth.
- **Dice Score**: A robust F1-based metric for segmentation quality.
- **Pixel Accuracy**: Overall pixel-wise classification performance.
- **Training and Validation Loss** (when applicable): Used to assess optimization behavior and generalization trends.

## Comparative Visualization

A grouped bar chart is used to visualize model performance across the selected metrics, enabling direct comparison under a unified experimental setting.  
This representation highlights both absolute performance differences and relative trade-offs between models of varying complexity.

## Key Observations

The comparative results reveal several important trends:
- High-capacity models such as **DeepLabV3** and **UNet** generally achieve stronger segmentation accuracy and overlap metrics.
- Lightweight architectures like **Fast-SCNN** and **ADA-Net** offer competitive performance with significantly reduced computational complexity.
- The classical **SVM** baseline provides a useful lower bound, illustrating the performance gap between traditional methods and deep learning approaches.
- Differences between training and validation loss offer insight into model generalization behavior across architectures.

This quantitative comparison provides a foundation for understanding the strengths, limitations, and practical trade-offs of each segmentation model, which are further discussed in the final analysis.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import re

# ===============================
# Configuration
# ===============================
# Define base path and training log files for each model
base_path = "/content/COMP9517"
model_files = {
    "SVM": os.path.join(base_path, "training_log_svm.csv"),
    "UNet": os.path.join(base_path, "training_log_unet.csv"),
    "DeepLabV3": os.path.join(base_path, "training_log_deeplabv3.csv"),
    "Fast-SCNN": os.path.join(base_path, "training_log_fastscnn.csv"),
    "ADA-Net": os.path.join(base_path, "training_log_adanet.csv")
}

# Map metric names in the code to the actual column names in each CSV
field_mappings = {
    "SVM": {"train_loss": None, "val_loss": None, "iou": "iou", "dice": "dice", "accuracy": "accuracy"},
    "UNet": {"train_loss": "train_loss", "val_loss": "val_loss", "iou": "iou", "dice": "dice", "accuracy": "accuracy"},
    "DeepLabV3": {"train_loss": "Train Loss", "val_loss": "Val Loss", "iou": "IoU", "dice": "Dice", "accuracy": "Accuracy"},
    "Fast-SCNN": {"train_loss": "Train Loss", "val_loss": "Val Loss", "iou": "IoU", "dice": "Dice", "accuracy": "Accuracy"},
    "ADA-Net": {"train_loss": "train_loss", "val_loss": "val_loss", "iou": "iou", "dice": "dice", "accuracy": "acc"}
}

# ===============================
# Extract final epoch metrics for each model
# ===============================
metrics = {}
for model_name, path in model_files.items():
    if not os.path.exists(path):
        print(f"[Missing file] {model_name}")
        continue
    df = pd.read_csv(path)
    if df.empty:
        print(f"[Empty file] {model_name}")
        continue

    # Get the last row (final epoch metrics)
    last_row = df.iloc[-1]
    model_metrics = {}
    for metric in ["train_loss", "val_loss", "iou", "dice", "accuracy"]:
        col = field_mappings[model_name].get(metric)
        if col and col in last_row:
            val = last_row[col]
            # If metric stored as tensor string, extract numeric value
            if isinstance(val, str) and "tensor" in val:
                match = re.search(r"[\d.]+", val)
                val = float(match.group()) if match else 0.0
            model_metrics[metric] = float(val)
        else:
            model_metrics[metric] = None  # mark as missing if not available
    metrics[model_name] = model_metrics

# ===============================
# Prepare data for plotting
# ===============================
model_names = list(metrics.keys())
train_loss = [metrics[m]["train_loss"] for m in model_names]
val_loss = [metrics[m]["val_loss"] for m in model_names]
iou = [metrics[m]["iou"] for m in model_names]
dice = [metrics[m]["dice"] for m in model_names]
acc = [metrics[m]["accuracy"] for m in model_names]

x = np.arange(len(model_names))
width = 0.15  # width of each bar in the group

# ===============================
# Create grouped bar chart
# ===============================
plt.figure(figsize=(14, 6))
bars1 = plt.bar(x - 2*width, [v if v is not None else 0 for v in train_loss], width, label="Train Loss")
bars2 = plt.bar(x - width, [v if v is not None else 0 for v in val_loss], width, label="Val Loss")
bars3 = plt.bar(x, [v if v is not None else 0 for v in iou], width, label="IoU")
bars4 = plt.bar(x + width, [v if v is not None else 0 for v in dice], width, label="Dice (F1)")
bars5 = plt.bar(x + 2*width, [v if v is not None else 0 for v in acc], width, label="Accuracy")

# Helper function to annotate bars with numeric values
def annotate_bars(bars, values):
    for bar, val in zip(bars, values):
        if val is not None:
            plt.annotate(f'{val:.3f}', xy=(bar.get_x() + bar.get_width() / 2, val),
                         xytext=(0, 3), textcoords="offset points",
                         ha='center', va='bottom', fontsize=8)

# Annotate each metric's bars
for bars, values in zip([bars1, bars2, bars3, bars4, bars5], [train_loss, val_loss, iou, dice, acc]):
    annotate_bars(bars, values)

# Configure plot appearance
plt.xticks(x, model_names)
plt.ylabel("Value")
plt.title("Final Epoch Metrics Comparison Across Models")
plt.ylim(0, 1.0)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()

# ===============================
# Save and display figure
# ===============================
output_dir = os.path.join(base_path, "outputs")
os.makedirs(output_dir, exist_ok=True)
save_path = os.path.join(output_dir, "final_epoch_comparison_all_metrics_bar_annotated.png")
plt.savefig(save_path)
plt.show()

print(f"Chart saved to: {save_path}")

### Training Dynamics Comparison Across Models

To complement the quantitative performance comparison, training dynamics of the four deep learning–based segmentation models (**UNet**, **DeepLabV3**, **Fast-SCNN**, and **ADA-Net**) are analyzed to assess convergence behavior and training stability.

Key metrics are tracked across training epochs, including training and validation loss, pixel accuracy, Intersection-over-Union (IoU), and Dice score.  
Comparing these trends over time provides insight into optimization efficiency, generalization behavior, and relative learning speed across architectures.

The resulting curves highlight differences in convergence rate, performance stability, and segmentation quality throughout training, offering a temporal perspective that complements final-epoch metric comparisons and supports deeper interpretation of model behavior.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

# ===============================
# Model training log file names
# ===============================
model_files = {
    "UNet": "training_log_unet.csv",
    "DeepLabV3": "training_log_deeplabv3.csv",
    "Fast-SCNN": "training_log_fastscnn.csv",
    "ADA-Net": "training_log_adanet.csv"
}

# ===============================
# Base and output directories
# ===============================
base_path = "/content/COMP9517"
output_dir = os.path.join(base_path, "outputs")
os.makedirs(output_dir, exist_ok=True)

# ===============================
# Metrics to plot for each model
# ===============================
metrics_to_plot = ["train_loss", "val_loss", "accuracy", "iou", "dice"]

# ===============================
# Column mappings for each model
# Maps original CSV column names to standardized names
# ===============================
column_mappings = {
    "UNet": {
        "epoch": "epoch",
        "train_loss": "train_loss",
        "val_loss": "val_loss",
        "iou": "iou",
        "dice": "dice",
        "accuracy": "accuracy"
    },
    "DeepLabV3": {
        "Epoch": "epoch",
        "Train Loss": "train_loss",
        "Val Loss": "val_loss",
        "IoU": "iou",
        "Dice": "dice",
        "Accuracy": "accuracy"
    },
    "Fast-SCNN": {
        "Epoch": "epoch",
        "Train Loss": "train_loss",
        "Val Loss": "val_loss",
        "IoU": "iou",
        "Dice": "dice",
        "Accuracy": "accuracy"
    },
    "ADA-Net": {
        "epoch": "epoch",
        "train_loss": "train_loss",
        "val_loss": "val_loss",
        "iou": "iou",
        "dice": "dice",
        "acc": "accuracy"
    }
}

# ===============================
# Fixed color mapping for plot lines
# ===============================
color_mapping = {
    "UNet": "#1f77b4",        # blue
    "DeepLabV3": "#ff7f0e",   # orange
    "Fast-SCNN": "#2ca02c",   # green
    "ADA-Net": "#d62728"      # red
}

# ===============================
# Load and preprocess each model's training log
# ===============================
model_data = {}

for model, filename in model_files.items():
    path = os.path.join(base_path, filename)
    if os.path.exists(path):
        df = pd.read_csv(path)

        # Rename columns according to mapping
        if model in column_mappings:
            df.rename(columns=column_mappings[model], inplace=True)

        # Convert dice values from 'tensor(...)' string format to float if needed
        if "dice" in df.columns:
            df["dice"] = df["dice"].apply(
                lambda x: float(str(x).replace("tensor(", "").split(",")[0]) if "tensor" in str(x) else float(x)
            )

        model_data[model] = df
    else:
        print(f"File not found for model {model}: {path}")

# ===============================
# Plot each metric across models
# ===============================
for metric in metrics_to_plot:
    plt.figure(figsize=(10, 6))

    for model, df in model_data.items():
        if metric in df.columns:
            x = df["epoch"]
            y = df[metric]
            plt.plot(x, y, label=model, color=color_mapping[model])
        else:
            print(f"[Warning] '{metric}' not found in model: {model}")

    plt.title(f"{metric.replace('_', ' ').title()} over Epochs")
    plt.xlabel("Epoch")
    plt.ylabel(metric.replace('_', ' ').title())
    plt.legend(loc="best")
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()

    # Save plot to output directory
    save_path = os.path.join(output_dir, f"{metric}_comparison_final.png")
    plt.savefig(save_path)
    print(f"Saved plot: {save_path}")
    plt.show()

## Discussion and Model Comparison Summary

This project implemented and evaluated five segmentation approaches—SVM, UNet, DeepLabV3 (ResNet-50 backbone), Fast-SCNN, and ADA-Net—for standing dead tree segmentation in aerial forest imagery.

### Quantitative Performance Comparison
Based on final-epoch evaluation metrics:
- **DeepLabV3** achieved the strongest overall performance, obtaining the highest accuracy, IoU, and Dice score, indicating superior capability in capturing multi-scale spatial context.
- **UNet** delivered consistently strong results with competitive overlap metrics, demonstrating that a well-designed encoder–decoder architecture remains effective for this task.
- **Fast-SCNN** achieved a favorable balance between performance and efficiency, converging quickly while maintaining reasonable segmentation quality.
- **ADA-Net**, despite its lightweight design, demonstrated stable learning behavior and outperformed the traditional SVM baseline, highlighting its potential as a compact segmentation model.
- **SVM**, as a classical machine learning approach, showed limited generalization ability on complex aerial imagery but served as a useful reference baseline.

### Training Dynamics Analysis
Analysis of training curves revealed clear differences in optimization behavior:
- **DeepLabV3** and **UNet** exhibited smooth and stable convergence with minimal signs of overfitting.
- **Fast-SCNN** converged rapidly but showed higher variance in validation performance, reflecting a trade-off between speed and stability.
- **ADA-Net** demonstrated gradual and consistent improvement across epochs, indicating reliable but slower convergence.

### Qualitative Evaluation
Qualitative inspection further supported the quantitative findings:
- **Worst-case prediction visualizations** exposed typical failure modes such as partial tree omission and confusion in densely vegetated regions.
- **Random sample predictions** confirmed that deep learning models generally produced coherent and spatially consistent segmentation masks under normal conditions.

### Overall Assessment
- **DeepLabV3** is the preferred choice when segmentation accuracy is the primary objective.
- **Fast-SCNN** is well-suited for scenarios requiring fast inference and low computational overhead.
- **UNet** provides a strong balance between architectural simplicity and segmentation quality.
- **ADA-Net** shows promise as a lightweight model that could benefit from further architectural refinement.

Overall, the comparative analysis demonstrates that no single model is optimal for all deployment scenarios. Model selection should be guided by practical constraints such as computational resources, inference speed requirements, and acceptable accuracy trade-offs.

## Final Packaging and Project Archival

To preserve the complete experimental workflow and results, all relevant project artifacts are packaged into a single archive for long-term storage and reproducibility.

The archived contents include:
- Trained model checkpoints for all evaluated methods
- Evaluation outputs such as metrics logs, prediction files, and visualization figures
- Training logs documenting optimization behavior and convergence
- Supporting scripts and configuration files required to reproduce the experiments

This packaging step ensures that the full project pipeline—from data preparation to final evaluation—can be reliably restored, reviewed, or extended in future work.

In [ ]:
import shutil
from google.colab import files

# ===============================
# Step 1: Create a ZIP archive of the folder
# ===============================
zip_path = '/content/COMP9517.zip'
shutil.make_archive('/content/COMP9517', 'zip', '/content/COMP9517')

# ===============================
# Step 2: Download the ZIP file to local machine
# ===============================
files.download(zip_path)

## Final Summary

This notebook presents a complete end-to-end workflow for standing dead tree segmentation from aerial imagery, covering model implementation, training, evaluation, and comparative analysis across multiple approaches.

Five segmentation methods were studied:
- **SVM** – A traditional machine learning baseline using handcrafted features
- **UNet** – A convolutional encoder–decoder architecture with skip connections
- **DeepLabV3** – A deep semantic segmentation model with atrous convolutions and a pretrained ResNet-50 backbone
- **Fast-SCNN** – A lightweight segmentation network optimized for real-time inference
- **ADA-Net** – A custom encoder–decoder architecture designed for efficiency and adaptability

The workflow includes:
- Dataset preparation using RGB and near-infrared (NIR) information with a fixed 80/20 training–validation split
- Consistent model training and evaluation using accuracy, IoU, Dice score, precision, and recall
- Visualization of training dynamics, prediction outputs, confusion matrices, and performance comparisons
- Quantitative and qualitative analysis highlighting the strengths and limitations of each model

All trained models, evaluation results, and visual artifacts are archived to support reproducibility and future extension.

Overall, this project demonstrates a modular and extensible segmentation pipeline that can be readily adapted to related remote sensing and environmental monitoring tasks, providing a practical reference for comparing classical machine learning methods with modern deep learning architectures.